In [1]:
#%pip install pandas numpy openai nest_asyncio scikit-learn tqdm

import random, asyncio, warnings
import datetime
from datetime import datetime, timezone
from typing import Any, Dict, Optional, Tuple, List
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import nest_asyncio
from openai import OpenAI, AsyncOpenAI

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from openai import OpenAI
#from rank_bm25 import BM25Okapi
import openpyxl
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter
import os
import json
import re

### Stratified Random Sampling of Political Speech Paragraphs

This cell creates a stratified random sample of a preferred size from the
political speech paragraph dataset.

Steps:
1. Load the merged and reviewed speech classification Excel file.
2. Filter out rows where color_source_code == 3 (unreliable/excluded sources).
3. Drop annotation-related columns that are not needed for fine-tuning.
4. Normalize party labels and clean paragraph text.
5. Clean and normalize the share_yes variable to a [0, 1] range.
6. Deduplicate rows based on paragraph text and party affiliation,
    keeping the most informative duplicate (highest n_rated, then share_yes).
7. Bin share_yes into 5 ordinal buckets aligned with the Ferenc 1–5 scale:
        0–19%  → 1  |  20–39% → 2  |  40–59% → 3  |  60–79% → 4  | 80–100% → 5
8. Draw a stratified random sample of SAMPLE_SIZE (default: 300) rows,
    proportional to each bin's share of the full dataset.
    Rounding adjustments ensure the final sample is exactly SAMPLE_SIZE rows.
9. Save the sampled DataFrame to an Excel file for downstream use
    (e.g., fine-tuning dataset construction, synthetic reasoning generation).

Configuration:
- SAMPLE_SIZE  : target number of rows in the output sample (default: 300)
- RANDOM_STATE : random seed for reproducibility (default: 42)
- INPUT        : path to the source Excel file
- OUTPUT_PATH  : path where the stratified sample will be saved


In [ ]:


# ================= CONFIGURATION =================
BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
INPUT = os.path.join(BASE_DIR, "input", "speech_classification_results_merged_review_COLORED_SORTED.xlsx")
OUTPUT_PATH = os.path.join(BASE_DIR, "output", "sample", "speech_sample_400_stratified.xlsx")

SAMPLE_SIZE = 400
RANDOM_STATE = 42

# ── 1. LOAD DATA ─────────────────────────────────────────────
df = pd.read_excel(INPUT)
print(f"Original N: {len(df)}")

# ── 2. FILTER OUT color_source_code == 3 ─────────────────────
df = df[df["color_source_code"] != 3].copy()
print(f"After removing color_source_code == 3: {len(df)}")

# ── 3. DROP UNNEEDED COLUMNS ─────────────────────────────────
cols_to_drop = [
    "too_vague", "unexplained_reference", "no_claim",
    "ambiguous_framing", "technical_jargon", "multiple_claims",
    "reason_too_vague", "reason_unexplained_reference",
    "reason_no_claim", "reason_ambiguous_framing",
    "reason_technical_jargon", "reason_multiple_claims",
    "flag_count", "flagged_any", "flagged_2plus",
    "Ask_ference"
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# Normalize party labels
if "party" not in df.columns:
    raise ValueError("party column not found.")
df["party"] = df["party"].replace({"Democrat": "Democratic"}).fillna("Unknown")

# Ensure paragraph_clean exists + normalize whitespace
if "paragraph_clean" not in df.columns:
    raise ValueError("paragraph_clean column not found.")
df["paragraph_clean"] = df["paragraph_clean"].astype(str).str.strip()

print("Remaining columns:", len(df.columns))

# ── 4. CLEAN share_yes ───────────────────────────────────────
if "share_yes" not in df.columns:
    raise ValueError("share_yes column not found.")

df["share_yes"] = pd.to_numeric(df["share_yes"], errors="coerce")
df["share_yes"] = np.where(df["share_yes"] > 1, df["share_yes"] / 100, df["share_yes"])
df["share_yes"] = df["share_yes"].clip(0, 1)

if df["share_yes"].isna().any():
    bad_n = int(df["share_yes"].isna().sum())
    raise ValueError(f"Some share_yes values are invalid (NA). Count={bad_n}")

# ── 4.5 DEDUPLICATE (paragraph_clean + party) ─────────────────
# Keep the most informative duplicate: max n_rated, then max share_yes
dedup_keys = ["paragraph_clean", "party"]

before = len(df)

if "n_rated" in df.columns:
    df["n_rated"] = pd.to_numeric(df["n_rated"], errors="coerce").fillna(0)
    df = (
        df.sort_values(dedup_keys + ["n_rated", "share_yes"], ascending=[True, True, False, False])
          .drop_duplicates(subset=dedup_keys, keep="first")
          .copy()
    )
else:
    df = df.drop_duplicates(subset=dedup_keys, keep="first").copy()

print(f"Dedup by {dedup_keys}: {before} -> {len(df)}")

# ── 5. CREATE SHARE BINS (5 bins aligned with 1–5 mapping) ──
bins = [0.0, 0.20, 0.40, 0.60, 0.80, 1.0000001]
labels = [1, 2, 3, 4, 5]

df["share_bin"] = pd.cut(
    df["share_yes"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

print("Bin distribution:")
print(df["share_bin"].value_counts().sort_index())

# ── 6. STRATIFIED SAMPLE OF 400 ─────────────────────────────
def _sample_group(x):
    n = int(np.round(len(x) / len(df) * SAMPLE_SIZE))
    n = min(max(n, 0), len(x))
    if n == 0:
        return x.iloc[0:0]
    return x.sample(n=n, random_state=RANDOM_STATE)

sampled_df = (
    df.groupby("share_bin", group_keys=False, observed=False)
      .apply(_sample_group)
      .copy()
)

# Adjust if rounding gave != SAMPLE_SIZE
if len(sampled_df) < SAMPLE_SIZE:
    remaining = SAMPLE_SIZE - len(sampled_df)
    remainder_pool = df.drop(sampled_df.index, errors="ignore")
    extra = remainder_pool.sample(
        n=min(remaining, len(remainder_pool)),
        random_state=RANDOM_STATE
    )
    sampled_df = pd.concat([sampled_df, extra], ignore_index=False)

if len(sampled_df) > SAMPLE_SIZE:
    sampled_df = sampled_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

# Ensure share_bin exists
if "share_bin" not in sampled_df.columns:
    sampled_df["share_bin"] = pd.cut(
        sampled_df["share_yes"],
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=False
    )

print(f"\nFinal sampled N: {len(sampled_df)}")
print("Sample bin distribution:")
print(sampled_df["share_bin"].value_counts().sort_index())

# ── 7. SAVE SAMPLE ───────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
sampled_df.drop(columns=["share_bin"], errors="ignore").to_excel(OUTPUT_PATH, index=False)

print(f"\nSaved stratified sample to:\n{OUTPUT_PATH}")

Original N: 551
After removing color_source_code == 3: 511
Remaining columns: 11
Dedup by ['paragraph_clean', 'party']: 511 -> 507
Bin distribution:
share_bin
1    132
2     77
3     94
4     81
5    123
Name: count, dtype: int64

Final sampled N: 400
Sample bin distribution:
share_bin
1    104
2     61
3     74
4     64
5     97
Name: count, dtype: int64

Saved stratified sample to:
C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\speech_sample_400_stratified.xlsx


### DW Collusion Fine-Tuning Dataset Builder

This cell constructs a supervised fine-tuning (SFT) dataset from the stratified
speech sample and exports it in three formats: **OpenAI**, **Gemini/Vertex**, and
**Open Models** (e.g., LLaMA, Mistral).

Steps:

1. **Load** the stratified sample Excel file (`speech_sample_300_stratified.xlsx`).
2. **Clean** party labels, normalize `share_yes` to `[0, 1]`, and drop unusable rows.
3. **Construct the ordinal target label** using Ferenc's 1–5 mapping:
    - 0–19% → 1 | 20–39% → 2 | 40–59% → 3 | 60–79% → 4 | 80–100% → 5
4. **Build prompt records** for each row using a shared system prompt and
    a user template that includes party affiliation and paragraph text.
    The assistant response is a JSON object: `{"score": <1–5>}`.
5. **Split** the dataset 80/20 into train and validation sets,
    stratified by score bucket.
6. **Write JSONL files** for each format:
    - `train_openai.jsonl` / `val_openai.jsonl`
    - `train_gemini.jsonl` / `val_gemini.jsonl`
    - `train_open.jsonl` / `val_open.jsonl`
7. **Write a README.md** summarizing the dataset, scaling logic, split sizes,
    and score distributions.

Configuration:
- `DATASET_FOLDER` : name of the output subdirectory
- `EXCEL_PATH`     : path to the stratified sample
- `OUTPUT_DIR`     : destination folder for all output files

In [ ]:
"""
DW Collusion Fine-Tuning Dataset Builder

Outputs 6 JSONL files:

OpenAI
 - train_openai.jsonl
 - val_openai.jsonl

Gemini
 - train_gemini.jsonl
 - val_gemini.jsonl

Open Models
 - train_open.jsonl
 - val_open.jsonl

Folder also includes:
README.md explaining the scaling and dataset.
"""



# =========================
# CONFIG
# =========================

BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"

DATASET_FOLDER = "finetune_dw_ordinal_scale_sample200"

OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", DATASET_FOLDER)
EXCEL_PATH = os.path.join(BASE_DIR, "input", "ft_sample", "speech_sample_200_stratified.xlsx")

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_OPENAI = os.path.join(OUTPUT_DIR, "train_openai.jsonl")
VAL_OPENAI   = os.path.join(OUTPUT_DIR, "val_openai.jsonl")

TRAIN_GEMINI = os.path.join(OUTPUT_DIR, "train_gemini.jsonl")
VAL_GEMINI   = os.path.join(OUTPUT_DIR, "val_gemini.jsonl")

TRAIN_OPEN   = os.path.join(OUTPUT_DIR, "train_open.jsonl")
VAL_OPEN     = os.path.join(OUTPUT_DIR, "val_open.jsonl")

README_PATH  = os.path.join(OUTPUT_DIR, "README.md")

# =========================
# LOAD DATA
# =========================

df = pd.read_excel(EXCEL_PATH)

required_cols = ["paragraph_clean", "party", "share_yes"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["party"] = df["party"].replace({"Democrat": "Democratic"}).fillna("Unknown")
df["paragraph_clean"] = df["paragraph_clean"].astype(str).str.strip()

df["share_yes"] = pd.to_numeric(df["share_yes"], errors="coerce")
df["share_yes"] = np.where(df["share_yes"] > 1, df["share_yes"] / 100, df["share_yes"])
df["share_yes"] = df["share_yes"].clip(0, 1)

# Drop unusable rows
df = df.dropna(subset=["share_yes"])
df = df[df["paragraph_clean"].ne("")].copy()

# =========================
# TARGET CONSTRUCTION
# Ferenc ordinal scaling:
# 0–19%   -> 1
# 20–39%  -> 2
# 40–59%  -> 3
# 60–79%  -> 4
# 80–100% -> 5
# =========================

bins   = [-0.001, 0.20, 0.40, 0.60, 0.80, 1.001]
labels = [1, 2, 3, 4, 5]

df["human_score_ferenc"] = pd.cut(
    df["share_yes"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
).astype(int)

print("Total usable examples:", len(df))
print("\nScore distribution:")
print(df["human_score_ferenc"].value_counts().sort_index())

# =========================
# PROMPTS
# =========================

SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{"score": <integer between 1 and 5 inclusive>}
""".strip()

USER_TEMPLATE = """
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# BUILD RECORDS
# =========================

openai_records = []
gemini_records = []
open_records   = []

for _, row in df.iterrows():
    user_msg = USER_TEMPLATE.format(
        party=row["party"],
        passage=row["paragraph_clean"]
    )

    target = json.dumps({"score": int(row["human_score_ferenc"])}, ensure_ascii=False)

    # OPENAI
    openai_records.append({
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]
    })

    # GEMINI
    gemini_records.append({
        "systemInstruction": {
            "role": "system",
            "parts": [{"text": SYSTEM_PROMPT}]
        },
        "contents": [
            {"role": "user",  "parts": [{"text": user_msg}]},
            {"role": "model", "parts": [{"text": target}]}
        ]
    })

    # OPEN MODELS
    open_records.append({
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]
    })

print("\nTotal records built:", len(open_records))

# =========================
# SPLIT DATA
# =========================

train_idx, val_idx = train_test_split(
    range(len(open_records)),
    test_size=0.20,
    random_state=42,
    stratify=df["human_score_ferenc"]
)

def subset(data, idx):
    return [data[i] for i in idx]

train_openai = subset(openai_records, train_idx)
val_openai   = subset(openai_records, val_idx)

train_gemini = subset(gemini_records, train_idx)
val_gemini   = subset(gemini_records, val_idx)

train_open = subset(open_records, train_idx)
val_open   = subset(open_records, val_idx)

print("Train size:", len(train_open))
print("Val size:",   len(val_open))

train_dist = df.iloc[list(train_idx)]["human_score_ferenc"].value_counts().sort_index().to_dict()
val_dist   = df.iloc[list(val_idx)]["human_score_ferenc"].value_counts().sort_index().to_dict()

print("\nTrain score distribution:", train_dist)
print("Val score distribution:", val_dist)

# =========================
# WRITE JSONL
# =========================

def write_jsonl(path, data):
    with open(path, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print("Saved:", path)

write_jsonl(TRAIN_OPENAI, train_openai)
write_jsonl(VAL_OPENAI,   val_openai)

write_jsonl(TRAIN_GEMINI, train_gemini)
write_jsonl(VAL_GEMINI,   val_gemini)

write_jsonl(TRAIN_OPEN, train_open)
write_jsonl(VAL_OPEN,   val_open)

# =========================
# WRITE README
# =========================

score_dist = df["human_score_ferenc"].value_counts().sort_index().to_dict()

readme = f"""
# DW Collusion Supervised Fine-Tuning Dataset

Dataset folder:
{DATASET_FOLDER}

## Purpose

Dataset for supervised fine-tuning of LLMs to predict perceived collusion
in political speech paragraphs.

## Input Data

Source file:
speech_sample_200_stratified.xlsx

Usable examples after cleaning:
{len(df)}

Each paragraph has a survey variable:

share_yes ∈ [0,1]

representing the share of respondents labeling the paragraph as conspiratorial.

## Score Scaling

This dataset uses Ferenc's ordinal scaling:

0–19%   → 1
20–39%  → 2
40–59%  → 3
60–79%  → 4
80–100% → 5

## Training Target

The model learns to predict:

{{"score": <int 1–5>}}

given party affiliation and paragraph text.

## Dataset Files

OpenAI
- train_openai.jsonl
- val_openai.jsonl

Gemini
- train_gemini.jsonl
- val_gemini.jsonl

Open Models
- train_open.jsonl
- val_open.jsonl

## Split

Train: {len(train_open)}
Validation: {len(val_open)}

Stratified by score.

## Full Score Distribution

{score_dist}

## Train Score Distribution

{train_dist}

## Validation Score Distribution

{val_dist}
""".strip()

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(readme)

print("Saved README:", README_PATH)

print("\nDataset folder ready:")
print(OUTPUT_DIR)

Total usable examples: 400

Score distribution:
human_score_ferenc
1    104
2     61
3     74
4     64
5     97
Name: count, dtype: int64

Total records built: 400
Train size: 320
Val size: 80

Train score distribution: {1: 83, 2: 49, 3: 59, 4: 51, 5: 78}
Val score distribution: {1: 21, 2: 12, 3: 15, 4: 13, 5: 19}
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\finetune_dw_ordinal_scale_sample400\train_openai.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\finetune_dw_ordinal_scale_sample400\val_openai.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\finetune_dw_ordinal_scale_sample400\train_gemini.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\finetune_dw_ordinal_scale_sample400\val_gemini.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\finetune_dw_ordinal_scale_sample400\train_open.jsonl


### OpenAI Fine-Tune Launcher for DW Ordinal Scale (Sample 300)

This cell launches an OpenAI supervised fine-tuning job using the stratified
300-sample dataset built in the previous cell.

Steps:
1. **Configure** paths, hyperparameters, and model settings.
2. **Check for an existing job** via a local JSON log file (`finetune_job_log_openai.json`):
    - If a job exists and is still running → resume and report its status.
    - If a job exists but is terminal (`succeeded`, `failed`, `cancelled`) → launch a new job.
    - If no log exists → proceed to upload and launch.
3. **Upload** train and validation JSONL files to the OpenAI Files API.
4. **Create** a fine-tuning job with the configured hyperparameters:
    - Model: `gpt-4.1-2025-04-14`
    - Epochs, batch size, and learning rate multiplier
    - Suffix: `dw_sample300_ordinal`
5. **Save** job metadata (job ID, file IDs, hyperparameters, timestamps) to the log file.
6. **Report** the initial job status and OpenAI UI link.

Configuration:
- `MODEL`      : base model to fine-tune
- `N_EPOCHS`   : number of training epochs
- `BATCH_SIZE` : training batch size
- `LR_MULT`    : learning rate multiplier
- `SUFFIX`     : fine-tuned model name suffix
- `JOB_LOG`    : path to the resume/log JSON file

In [ ]:
"""
OpenAI Fine-Tune Launcher (no monitoring, notebook-friendly)
===========================================================
- Auto-resumes from job log if interrupted
- Launches a new job if the previous one is terminal
- No polling / no progress bar / no SystemExit
"""
# ── CONFIG ─────────────────────────────────────────────────────────────────────

BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"

DATASET_FOLDER = "finetune_dw_ordinal_scale_sample400"

OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", DATASET_FOLDER)

TRAIN_PATH = os.path.join(OUTPUT_DIR, "train_openai.jsonl")
VAL_PATH   = os.path.join(OUTPUT_DIR, "val_openai.jsonl")

JOB_LOG = os.path.join(OUTPUT_DIR, "finetune_job_log_openai.json")

MODEL = "gpt-4.1-2025-04-14"

# Hyperparameters
N_EPOCHS   = 3
BATCH_SIZE = 4
LR_MULT    = 0.5

# Model suffix (important for distinguishing experiments)
SUFFIX = "dw_collusion_ordinal_survey400"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# ── HELPERS ────────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def load_log():
    try:
        return json.load(open(JOB_LOG, encoding="utf-8")) if os.path.exists(JOB_LOG) else {}
    except Exception:
        return {}

def save_log(data):
    os.makedirs(os.path.dirname(JOB_LOG), exist_ok=True)
    with open(JOB_LOG, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)


# ── LAUNCH OR RESUME ───────────────────────────────────────────────────────────

job_log = load_log()
job_id  = job_log.get("job_id")

if job_id:
    existing = client.fine_tuning.jobs.retrieve(job_id)

    if existing.status in ("succeeded", "failed", "cancelled"):
        log(f"Previous job {job_id} already {existing.status} — launching new job.")
        job_id = None

    else:
        log(f"Resuming existing job {job_id} (status={existing.status})")
        log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

        save_log({
            **job_log,
            "status": existing.status,
            "resumed_at": datetime.now().isoformat()
        })

        job_id = existing.id


# ── START NEW JOB ──────────────────────────────────────────────────────────────

if not job_id:

    train_n = count_lines(TRAIN_PATH)
    val_n   = count_lines(VAL_PATH)

    log(f"Train rows: {train_n}  |  Val rows: {val_n}")

    log("Uploading files...")

    with open(TRAIN_PATH, "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")

    with open(VAL_PATH, "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")

    log(f"Train file id: {train_file.id}")
    log(f"Val file id:   {val_file.id}")

    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=MODEL,
        hyperparameters={
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate_multiplier": LR_MULT
        },
        suffix=SUFFIX
    )

    job_id = job.id

    log(f"✅ Job launched: {job_id}")
    log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

    save_log({
        "job_id": job_id,
        "model": MODEL,
        "train_rows": train_n,
        "val_rows": val_n,
        "train_file_id": train_file.id,
        "val_file_id": val_file.id,
        "hyperparameters": {
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr_mult": LR_MULT
        },
        "suffix": SUFFIX,
        "launched_at": datetime.now().isoformat(),
        "status": job.status
    })


# ── CHECK INITIAL STATUS ───────────────────────────────────────────────────────

latest = client.fine_tuning.jobs.retrieve(job_id)

log(f"Initial status: {latest.status}")
log("Done. You can run other code now.")

[16:46:21] Train rows: 320  |  Val rows: 80
[16:46:21] Uploading files...
[16:46:22] Train file id: file-7DZ9dHu3vzCssWraVwBiKY
[16:46:22] Val file id:   file-Bh386hh5EARU9UGx6q7qRZ
[16:46:25] ✅ Job launched: ftjob-Qg3ErlVLNwsFUDC0Rdzq4Wx0
[16:46:25] OpenAI UI: https://platform.openai.com/finetune/ftjob-Qg3ErlVLNwsFUDC0Rdzq4Wx0
[16:46:25] Initial status: validating_files
[16:46:25] Done. You can run other code now.


### DW Multi-Model Inference Runner for Political Speech Paragraphs

This cell runs multiple **OpenAI fine-tuned DW models sequentially** on the same
political speech dataset and writes all model outputs back to a single output file.

Steps:
1. Load the input spreadsheet containing political speech paragraphs.
2. Configure one or more enabled fine-tuned models in `MODEL_CFG`, each with:
    - model ID
    - score output column
    - reasoning output column
3. Resume safely from an existing output file, merging previously completed model
   predictions back into the working DataFrame when possible.
4. Build a shared DW prompt for each paragraph using party affiliation and text.
5. Run asynchronous inference in batches for each enabled model, while:
    - respecting concurrency limits
    - retrying transient API failures
    - attempting JSON extraction and lightweight repair
    - optionally issuing a second repair call if the model output is invalid
6. Save per-row failure/repair logs to a timestamped run folder for debugging.
7. Write model scores and reasons into model-specific columns in the output file
   after each batch as a checkpoint.
8. Print an end-of-run summary showing usable outputs, repairs, failures,
   and API errors for each model.

Configuration:
- `INPUT_FILE`              : source Excel file to score
- `OUTPUT_FILE`             : destination Excel file with appended model outputs
- `OPENAI_KEY_PATH`         : path to the OpenAI API key text file
- `MODEL_CFG`               : dictionary of enabled fine-tuned models and target columns
- `CONCURRENCY`             : number of simultaneous API requests
- `BATCH_SIZE`              : number of rows processed per save checkpoint
- `RESUME_IF_OUTPUT_EXISTS` : whether to merge and continue from prior output
- `ENABLE_REPAIR_CALL`      : whether to send a second repair prompt for invalid JSON
- `REASONING_REQUIRED`      : whether the model must return explanatory reasoning
- `DEBUG_TEST_ONLY`         : if `True`, run only a small test subset
- `DEBUG_N_TEST`            : number of rows to process in test mode
- `MAX_TOKENS`              : maximum tokens per response
- `TEMPERATURE`             : sampling temperature for inference output

Outputs:
- Updated spreadsheet with one score/reason column pair per enabled model
- Timestamped JSON debug logs for repaired, failed, or API-error rows
- Printed summary of run quality by model

In [ ]:
# ============================================================
# DW-only — Run MANY OpenAI fine-tuned models sequentially
# - Config-driven via MODEL_CFG
# - Resume-safe
# - JSON repair
# - Separate log subfolder per run
# - End-of-run summary
# ============================================================

warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"

INPUT_FILE  = os.path.join(BASE_DIR, "input",  "speech_classification_results_finetuned_dw_20_models.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "output", "speech_classification_results_finetuned_dw_multi_models.xlsx")
OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

TEMP_DIR = os.path.join(BASE_DIR, "temp", "_openai_inference_logs")

RUN_TS = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S")
RUN_LOG_DIR = os.path.join(TEMP_DIR, f"run_{RUN_TS}")
os.makedirs(RUN_LOG_DIR, exist_ok=True)

CONCURRENCY = 2
BATCH_SIZE  = 80

RESUME_IF_OUTPUT_EXISTS = True
ENABLE_REPAIR_CALL      = True

REASONING_REQUIRED = "YES"

DEBUG_TEST_ONLY = False
DEBUG_N_TEST    = 10
TEST_OUTPUT_SUFFIX = "_TEST"

MAX_TOKENS = 600
TEMPERATURE = 0

# =========================
# MODEL CONFIG
# Add as many models as you want here
# enabled=False skips that model
# =========================
MODEL_CFG = {
    "ft300": {
        "enabled": True,
        "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-survey300:DK5HnsoN",
        "score_col": "DW_gpt41_ft300_score",
        "reason_col": "DW_gpt41_ft300_reason",
    },
    "ft400": {
        "enabled": True,
        "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-survey400:DK5NO2xm",
        "score_col": "DW_gpt41_ft400_score",
        "reason_col": "DW_gpt41_ft400_reason",
    },

    # Example:
    # "ft500": {
    #     "enabled": False,
    #     "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-survey500:XXXX",
    #     "score_col": "DW_gpt41_ft500_score",
    #     "reason_col": "DW_gpt41_ft500_reason",
    # },
}

def safe_test_output_path(path: str) -> str:
    root, ext = os.path.splitext(path)
    return f"{root}{TEST_OUTPUT_SUFFIX}{ext}"

# =========================
# Load API key + client
# =========================
with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()
if not OPENAI_API_KEY:
    raise RuntimeError("openai_api.txt is empty")

client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content. Be attentive to party dynamics in such cases.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{
"score": <integer between 1 and 5 inclusive>, "reason": <string or null>
}

RULES FOR "reason":
- You will be told whether REASONING is REQUIRED via the user message.
- If REASONING IS REQUIRED: set "reason" to a 3-5 sentence explanation that:
  - Includes at least one direct quoted fragment from the passage.
  - Explicitly connects the quoted language to the rubric criteria for DW.
  - Briefly explains why a higher or lower score was not chosen.
- If REASONING IS NOT REQUIRED: "reason" must be exactly null (not the string "null", not "", not "N/A").
""".strip()

USER_TEMPLATE = """
REASONING_REQUIRED: {reasoning_required}
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# RUN STATS
# =========================
RUN_STATS = {name: Counter() for name in MODEL_CFG.keys()}

# =========================
# Logging
# =========================
def save_inference_log(
    model_name: str,
    row_idx: int,
    user_msg: str,
    raw_output: str,
    status: str,  # repaired | failed | api_error
    broken_output: Optional[str] = None,
    note: Optional[str] = None,
) -> None:
    ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S")
    obj = {
        "model": model_name,
        "row_index": row_idx,
        "timestamp_utc": ts,
        "status": status,
        "user_prompt": user_msg,
        "raw_output": raw_output,
    }
    if broken_output is not None:
        obj["broken_output"] = broken_output
    if note is not None:
        obj["note"] = note

    fname = f"{model_name}_row{row_idx:05d}_{status}_{ts}.json"
    with open(os.path.join(RUN_LOG_DIR, fname), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# =========================
# Helpers
# =========================
def _read_tabular(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm", ".xlsb"):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def save_full(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def build_user_message(row: pd.Series) -> str:
    party = (str(row.get("party", "")) or "UNKNOWN").strip()
    passage = str(row.get("paragraph_clean", "")).strip()
    return USER_TEMPLATE.format(
        reasoning_required=REASONING_REQUIRED,
        party=party,
        passage=passage
    )

def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = str(text).strip()

    if t.startswith("```"):
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
        if t.startswith("json"):
            t = t[4:]
        t = t.strip()

    try:
        obj = json.loads(t)
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        pass

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None

    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        return None

def code_repair_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    t = re.sub(r"```json|```", "", str(text)).strip()
    m = re.search(r"\{.*", t, flags=re.DOTALL)
    if not m:
        return None

    t = m.group(0)
    t = re.sub(r",\s*([}\]])", r"\1", t)

    if t.count('"') % 2 != 0:
        t += '"'
    t += "}" * max(0, t.count("{") - t.count("}"))
    t += "]" * max(0, t.count("[") - t.count("]"))

    try:
        return json.loads(t)
    except Exception:
        pass

    score_m  = re.search(r'"score"\s*:\s*([1-5])', t)
    reason_m = re.search(r'"reason"\s*:\s*"(.*?)(?:"|$)', t, flags=re.DOTALL)
    if score_m:
        return {
            "score": int(score_m.group(1)),
            "reason": reason_m.group(1).strip() if reason_m else None,
        }
    return None

def validate_dw_reason(obj: Dict[str, Any]) -> Tuple[bool, str, Optional[int], Optional[str]]:
    if not isinstance(obj, dict):
        return False, "Not a dict", None, None

    if "score" not in obj or "reason" not in obj:
        return False, "Missing score/reason", None, None

    s = obj["score"]
    r = obj["reason"]

    if isinstance(s, str) and s.strip().isdigit():
        s = int(s.strip())

    if not isinstance(s, int) or not (1 <= s <= 5):
        return False, f"Bad score: {s!r}", None, None

    if REASONING_REQUIRED == "YES":
        if not isinstance(r, str) or len(r.strip()) < 10:
            return False, f"Bad reason: {type(r).__name__}", None, None
        return True, "OK", s, r.strip()

    return True, "OK", s, None

async def with_retries(call_fn, meta: str, max_retries=10, base_wait=2, max_wait=120):
    last_err = None
    for i in range(max_retries):
        try:
            return await call_fn()
        except Exception as e:
            last_err = e
            msg = str(e)
            low = msg.lower()

            is_429 = "429" in low or "rate limit" in low
            is_transient = (
                is_429 or
                "503" in low or "overload" in low or "unavailable" in low or
                "timeout" in low or "timed out" in low or "deadline" in low or
                "temporarily" in low or "try again" in low
            )
            if not is_transient:
                return f"Error:{msg[:300]}"

            base = 15 if is_429 else base_wait
            wait = min(max_wait, base * (2 ** i)) + random.uniform(0, 1.5)
            print(f"\n⚠️ Transient error ({meta}): {msg[:120]} | retry in {wait:.1f}s")
            await asyncio.sleep(wait)

    return f"Error:MaxRetries:{meta}:{str(last_err)[:200]}"

# =========================
# Model helpers
# =========================
def enabled_models() -> Dict[str, Dict[str, str]]:
    return {k: v for k, v in MODEL_CFG.items() if v.get("enabled", True)}

def all_output_cols() -> List[str]:
    cols = []
    for cfg in MODEL_CFG.values():
        cols.extend([cfg["score_col"], cfg["reason_col"]])
    return cols

def ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in all_output_cols():
        if c not in df.columns:
            df[c] = ""
    return df

def row_done(df: pd.DataFrame, i: int, model_name: str) -> bool:
    cfg = MODEL_CFG[model_name]
    score_col = cfg["score_col"]
    reason_col = cfg["reason_col"]
    return (
        str(df.at[i, score_col]).strip() != "" and
        str(df.at[i, reason_col]).strip() != ""
    )

def resume_merge(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    if not os.path.exists(out_path):
        return ensure_cols(df)

    old = _read_tabular(out_path)
    old = ensure_cols(old)
    df  = ensure_cols(df)

    key = None
    for k in ["q_id", "ParagraphID"]:
        if k in df.columns and k in old.columns:
            key = k
            break
    if key is None:
        return df

    df[key]  = df[key].astype(str).str.strip()
    old[key] = old[key].astype(str).str.strip()

    old = old.sort_values(key).drop_duplicates(key, keep="last")

    keep_cols = [key] + all_output_cols()
    old = old[[c for c in keep_cols if c in old.columns]]

    df = df.merge(old, on=key, how="left", suffixes=("", "_old"))

    for c in all_output_cols():
        oldc = f"{c}_old"
        if oldc in df.columns:
            df[c] = df[c].where(df[c].astype(str).str.strip() != "", df[oldc].fillna(""))
            df.drop(columns=[oldc], inplace=True)

    return df

# =========================
# Inference
# =========================
async def score_dw_with_reason(
    model_name: str,
    row: pd.Series,
    sema: asyncio.Semaphore,
    meta: str,
    row_idx: int,
) -> Dict[str, Any]:
    async with sema:
        user_msg = build_user_message(row)
        model_id = MODEL_CFG[model_name]["model"]

        async def _primary():
            resp = await client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            return (resp.choices[0].message.content or "").strip()

        res = await with_retries(_primary, meta=meta)
        if isinstance(res, str) and res.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "api_error")
            return {"score": None, "reason": res}

        obj = extract_json_from_text(res) or code_repair_json(res)
        if obj is not None:
            ok, _, s, r = validate_dw_reason(obj)
            if ok:
                RUN_STATS[model_name]["ok"] += 1
                return {"score": s, "reason": r}

        if not ENABLE_REPAIR_CALL:
            RUN_STATS[model_name]["failed"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "failed", note="No repair call enabled.")
            return {"score": None, "reason": "INVALID_JSON_NO_REPAIR"}

        obj_pre = extract_json_from_text(res) or code_repair_json(res)
        fail_reason = validate_dw_reason(obj_pre)[1] if obj_pre is not None else "unparseable JSON"

        repair_system = (
            "Fix the following output into VALID JSON ONLY.\n"
            'Required format: {"score": 3, "reason": "3-5 sentences with a direct quote fragment."}\n'
            "No markdown. No extra text. No extra keys."
        )
        repair_user = (
            f"Your previous output failed validation. Reason: {fail_reason}\n\n"
            f"Broken output:\n{res}"
        )

        async def _repair():
            resp = await client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": repair_system},
                    {"role": "user",   "content": repair_user},
                ],
                temperature=0,
                max_tokens=MAX_TOKENS,
            )
            return (resp.choices[0].message.content or "").strip()

        res2 = await with_retries(_repair, meta=meta + "|repair")
        if isinstance(res2, str) and res2.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res2, "api_error", broken_output=res)
            return {"score": None, "reason": res2}

        obj2 = extract_json_from_text(res2) or code_repair_json(res2)
        if obj2 is not None:
            ok2, _, s2, r2 = validate_dw_reason(obj2)
            if ok2:
                RUN_STATS[model_name]["repaired"] += 1
                save_inference_log(
                    model_name, row_idx, user_msg, res2, "repaired",
                    broken_output=res,
                    note="Primary response failed validation and was repaired."
                )
                return {"score": s2, "reason": r2}

        RUN_STATS[model_name]["failed"] += 1
        save_inference_log(
            model_name, row_idx, user_msg, res, "failed",
            note="Failed after repair attempt."
        )
        return {"score": None, "reason": "INVALID_JSON_AFTER_REPAIR"}

# =========================
# Sequential runner
# =========================
async def run_one_model(
    df: pd.DataFrame,
    model_name: str,
    sema: asyncio.Semaphore,
    process_indices: List[int],
) -> None:
    cfg = MODEL_CFG[model_name]
    score_col  = cfg["score_col"]
    reason_col = cfg["reason_col"]

    for k in range(0, len(process_indices), BATCH_SIZE):
        batch = process_indices[k:k + BATCH_SIZE]
        todo  = [i for i in batch if not row_done(df, i, model_name)]
        if not todo:
            continue

        async def run_row(ii: int):
            meta = f"{model_name}|row={ii}"
            obj = await score_dw_with_reason(model_name, df.iloc[ii], sema, meta, ii)
            return ii, obj

        tasks = [asyncio.create_task(run_row(i)) for i in todo]

        batch_num = k // BATCH_SIZE + 1
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"{model_name} batch {batch_num}"):
            i, obj = await fut
            score = obj.get("score")
            reason = obj.get("reason")

            df.at[i, score_col]  = "" if score is None else str(score)
            df.at[i, reason_col] = "" if reason is None else str(reason)

        save_full(df, OUTPUT_PATH)
        print(f"[Checkpoint] Saved → {OUTPUT_PATH}")

# =========================
# Summary
# =========================
def print_summary():
    print("\n================ SUMMARY ================")
    for model_name, cfg in enabled_models().items():
        stats = RUN_STATS[model_name]
        ok = stats.get("ok", 0)
        repaired = stats.get("repaired", 0)
        failed = stats.get("failed", 0)
        api_error = stats.get("api_error", 0)
        total_processed = ok + repaired + failed + api_error

        print(f"{model_name}:")
        print(f"  model       = {cfg['model']}")
        print(f"  processed   = {total_processed}")
        print(f"  ok          = {ok}")
        print(f"  repaired    = {repaired}")
        print(f"  failed      = {failed}")
        print(f"  api_error   = {api_error}")
        print(f"  usable      = {ok + repaired}")
        print()

    print(f"Log folder: {RUN_LOG_DIR}")
    print("=========================================\n")

# =========================
# MAIN
# =========================
async def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(INPUT_FILE)

    global OUTPUT_PATH
    OUTPUT_PATH = safe_test_output_path(OUTPUT_FILE) if DEBUG_TEST_ONLY else OUTPUT_FILE

    df = _read_tabular(INPUT_FILE)
    df = ensure_cols(df)

    if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_PATH):
        df = resume_merge(df, OUTPUT_PATH)

    for c in ["paragraph_clean", "party"]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    active_models = enabled_models()
    if not active_models:
        raise ValueError("No models enabled in MODEL_CFG.")

    n = len(df)
    process_indices = list(range(min(DEBUG_N_TEST, n))) if DEBUG_TEST_ONLY else list(range(n))
    sema = asyncio.Semaphore(CONCURRENCY)

    print(f"Rows: {n:,} | Processing: {len(process_indices):,}")
    print(f"Output: {OUTPUT_PATH}")
    print(f"Logs root: {TEMP_DIR}")
    print(f"Run logs:  {RUN_LOG_DIR}")
    print(f"Test mode: {DEBUG_TEST_ONLY}")
    print(f"Enabled models: {list(active_models.keys())}")

    for model_name, cfg in active_models.items():
        print(f"\n=== Running {model_name} sequentially ===")
        print(cfg["model"])
        await run_one_model(df, model_name, sema, process_indices)

    save_full(df, OUTPUT_PATH)
    print_summary()
    print("✅ DONE")

# RUN
await main()

Rows: 551 | Processing: 551
Output: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_models.xlsx
Logs root: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\_openai_inference_logs
Run logs:  C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\_openai_inference_logs\run_20260316T181613
Test mode: False
Enabled models: ['ft300', 'ft400']

=== Running ft300 sequentially ===
ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-survey300:DK5HnsoN


ft300 batch 1: 100%|██████████| 80/80 [04:17<00:00,  3.21s/it]


[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_models.xlsx


ft300 batch 2:   0%|          | 0/80 [00:00<?, ?it/s]

### Prompt with Examples

In [ ]:
# ============================================================
# DW-only — Run MANY OpenAI fine-tuned models sequentially
# - Config-driven via MODEL_CFG
# - Resume-safe
# - JSON repair
# - Separate log subfolder per run
# - End-of-run summary
# ============================================================

warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

# =========================
# CONFIG
# =========================
#BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
BASE_DIR = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"

INPUT_FILE  = os.path.join(BASE_DIR, "output", "speech_classification_results_finetuned_dw_multi_models.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "output", "speech_classification_results_finetuned_dw_multi_example_models.xlsx")
OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

TEMP_DIR = os.path.join(BASE_DIR, "temp", "_openai_inference_logs")

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
RUN_LOG_DIR = os.path.join(TEMP_DIR, f"run_{RUN_TS}")
os.makedirs(RUN_LOG_DIR, exist_ok=True)

CONCURRENCY = 2
BATCH_SIZE  = 80

RESUME_IF_OUTPUT_EXISTS = True
ENABLE_REPAIR_CALL      = True

REASONING_REQUIRED = "YES"

DEBUG_TEST_ONLY = False
DEBUG_N_TEST    = 10
TEST_OUTPUT_SUFFIX = "_TEST"

MAX_TOKENS = 800
TEMPERATURE = 0

# =========================
# MODEL CONFIG
# =========================
MODEL_CFG = {
    "gpt41_fs": {
        "enabled": True,
        "model": "gpt-4.1-2025-04-14",
        "score_col": "DW_gpt41_example_score",
        "reason_col": "DW_gpt41_example_reason",
    },
    "gpt5_mini_fs": {
        "enabled": True,
        "model": "gpt-5.4-mini-2026-03-17",
        "score_col": "DW_gpt5_mini_example_score",
        "reason_col": "DW_gpt5_mini_example_reason",
    },
    "gpt5_fs": {
        "enabled": True,
        "model": "gpt-5.4-2026-03-05",
        "score_col": "DW_gpt5_example_score",
        "reason_col": "DW_gpt5_example_reason",
    },

}


def safe_test_output_path(path: str) -> str:
    root, ext = os.path.splitext(path)
    return f"{root}{TEST_OUTPUT_SUFFIX}{ext}"

# =========================
# Load API key + client
# =========================
with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()
if not OPENAI_API_KEY:
    raise RuntimeError("openai_api.txt is empty")

client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content. Be attentive to party dynamics in such cases.

### Scored Examples:

Passage: "We are not on a witch hunt. We have the facts of the case, and we believe the right thing to do is to offer some remedy."
Speaker party: Republican
Score: 1

Passage: "That means additional unemployment benefits to help workers pay their bills while they look for new jobs. It means transitional health coverage, so workers don't lose their insurance if they lose their jobs. It means assistance to states, so they aren't forced to cut their budgets for critical necessities like Medicaid."
Speaker party: Democratic
Score: 1

Passage: "In recent days, Mr. Trump has stepped up talk of a potentially 'rigged' election, earning him rebukes from Republicans and Democrats alike."
Speaker party: Republican
Score: 2

Passage: "You know, for this reason, Democrats, the swamp, and some RINOs banded together to oppose Pete Hegseth immediately after he was nominated by President Trump. I am convinced that their opposition to Hegseth was only because he was not part of the military-industrial complex."
Speaker party: Republican
Score: 3

Passage: "So whether you have used Special Counsel Mueller as a patriot conducting a nonpartisan investigation into a foreign power's possible influence in our elections or as a witch hunt, a full accounting and public release of the findings is needed to heal our political differences."
Speaker party: Democratic
Score: 3

Passage: "Republicans wrote this resolution. Democrats were not consulted or allowed any input. Republicans then rigged the process and outlawed any amendments, so that the only real choice in voting was not yea or nay, but take it or leave it."
Speaker party: Democratic
Score: 3

Passage: "In the 2016 Presidential campaign, you had two candidates with wildly different solutions, but their fundamental diagnosis was the same: 'The system is rigged; you're getting screwed; you're a victim; this country is going down the tubes.'"
Speaker party: Republican
Score: 4

Passage: "Sinclair Broadcasting owns 62 Fox, ABC, CBS, NBC, WB and UPN affiliates which reach 24 percent of U.S. households. Its decision to blatantly use public airwaves for political use, to attempt to cover-up that attack as 'news programming,' and to disingenuously invite Senator Kerry to a panel discussion to meet fairness requirements would seem to violate current law and regulation."
Speaker party: Democratic
Score: 4

Passage: "These national emergency declarations have become the deep state's favorite foreign affairs slush fund. There is literally no spending transparency whatsoever out there on the 41 national emergencies currently in place. Congress is allowing the swamp to continue to shift and funnel an unknown amount of taxpayer dollars to an overseas dark hole."
Speaker party: Republican
Score: 5

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{
"score": <integer between 1 and 5 inclusive>, "reason": <string or null>
}

RULES FOR "reason":
- You will be told whether REASONING is REQUIRED via the user message.
- If REASONING IS REQUIRED: set "reason" to a 3-5 sentence explanation that:
  - Includes at least one direct quoted fragment from the passage.
  - Explicitly connects the quoted language to the rubric criteria for DW.
  - Briefly explains why a higher or lower score was not chosen.
- If REASONING IS NOT REQUIRED: "reason" must be exactly null (not the string "null", not "", not "N/A").
""".strip()


USER_TEMPLATE = """
REASONING_REQUIRED: {reasoning_required}
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# RUN STATS
# =========================
RUN_STATS = {name: Counter() for name in MODEL_CFG.keys()}

# =========================
# Logging
# =========================
def save_inference_log(
    model_name: str,
    row_idx: int,
    user_msg: str,
    raw_output: str,
    status: str,  # repaired | failed | api_error
    broken_output: Optional[str] = None,
    note: Optional[str] = None,
) -> None:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    obj = {
        "model": model_name,
        "row_index": row_idx,
        "timestamp_utc": ts,
        "status": status,
        "user_prompt": user_msg,
        "raw_output": raw_output,
    }
    if broken_output is not None:
        obj["broken_output"] = broken_output
    if note is not None:
        obj["note"] = note

    fname = f"{model_name}_row{row_idx:05d}_{status}_{ts}.json"
    with open(os.path.join(RUN_LOG_DIR, fname), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# =========================
# Helpers
# =========================
def _read_tabular(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm", ".xlsb"):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def save_full(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def build_user_message(row: pd.Series) -> str:
    party = (str(row.get("party", "")) or "UNKNOWN").strip()
    passage = str(row.get("paragraph_clean", "")).strip()
    return USER_TEMPLATE.format(
        reasoning_required=REASONING_REQUIRED,
        party=party,
        passage=passage
    )

def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = str(text).strip()

    if t.startswith("```"):
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
        if t.startswith("json"):
            t = t[4:]
        t = t.strip()

    try:
        obj = json.loads(t)
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        pass

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None

    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        return None

def code_repair_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    t = re.sub(r"```json|```", "", str(text)).strip()
    m = re.search(r"\{.*", t, flags=re.DOTALL)
    if not m:
        return None

    t = m.group(0)
    t = re.sub(r",\s*([}\]])", r"\1", t)

    if t.count('"') % 2 != 0:
        t += '"'
    t += "}" * max(0, t.count("{") - t.count("}"))
    t += "]" * max(0, t.count("[") - t.count("]"))

    try:
        return json.loads(t)
    except Exception:
        pass

    score_m  = re.search(r'"score"\s*:\s*([1-5])', t)
    reason_m = re.search(r'"reason"\s*:\s*"(.*?)(?:"|$)', t, flags=re.DOTALL)
    if score_m:
        return {
            "score": int(score_m.group(1)),
            "reason": reason_m.group(1).strip() if reason_m else None,
        }
    return None

def validate_dw_reason(obj: Dict[str, Any]) -> Tuple[bool, str, Optional[int], Optional[str]]:
    if not isinstance(obj, dict):
        return False, "Not a dict", None, None

    if "score" not in obj or "reason" not in obj:
        return False, "Missing score/reason", None, None

    s = obj["score"]
    r = obj["reason"]

    if isinstance(s, str) and s.strip().isdigit():
        s = int(s.strip())

    if not isinstance(s, int) or not (1 <= s <= 5):
        return False, f"Bad score: {s!r}", None, None

    if REASONING_REQUIRED == "YES":
        if not isinstance(r, str) or len(r.strip()) < 10:
            return False, f"Bad reason: {type(r).__name__}", None, None
        return True, "OK", s, r.strip()

    return True, "OK", s, None

async def with_retries(call_fn, meta: str, max_retries=10, base_wait=2, max_wait=120):
    last_err = None
    for i in range(max_retries):
        try:
            return await call_fn()
        except Exception as e:
            last_err = e
            msg = str(e)
            low = msg.lower()

            is_429 = "429" in low or "rate limit" in low
            is_transient = (
                is_429 or
                "503" in low or "overload" in low or "unavailable" in low or
                "timeout" in low or "timed out" in low or "deadline" in low or
                "temporarily" in low or "try again" in low
            )
            if not is_transient:
                return f"Error:{msg[:300]}"

            base = 15 if is_429 else base_wait
            wait = min(max_wait, base * (2 ** i)) + random.uniform(0, 1.5)
            print(f"\n⚠️ Transient error ({meta}): {msg[:120]} | retry in {wait:.1f}s")
            await asyncio.sleep(wait)

    return f"Error:MaxRetries:{meta}:{str(last_err)[:200]}"

# =========================
# Model helpers
# =========================
def enabled_models() -> Dict[str, Dict[str, str]]:
    return {k: v for k, v in MODEL_CFG.items() if v.get("enabled", True)}

def all_output_cols() -> List[str]:
    cols = []
    for cfg in MODEL_CFG.values():
        cols.extend([cfg["score_col"], cfg["reason_col"]])
    return cols

def ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in all_output_cols():
        if c not in df.columns:
            df[c] = ""
    return df

def row_done(df: pd.DataFrame, i: int, model_name: str) -> bool:
    cfg = MODEL_CFG[model_name]
    score_col = cfg["score_col"]
    reason_col = cfg["reason_col"]
    return (
        str(df.at[i, score_col]).strip() != "" and
        str(df.at[i, reason_col]).strip() != ""
    )

def resume_merge(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    if not os.path.exists(out_path):
        return ensure_cols(df)

    old = _read_tabular(out_path)
    old = ensure_cols(old)
    df  = ensure_cols(df)

    key = None
    for k in ["q_id", "ParagraphID"]:
        if k in df.columns and k in old.columns:
            key = k
            break
    if key is None:
        return df

    df[key]  = df[key].astype(str).str.strip()
    old[key] = old[key].astype(str).str.strip()

    old = old.sort_values(key).drop_duplicates(key, keep="last")

    keep_cols = [key] + all_output_cols()
    old = old[[c for c in keep_cols if c in old.columns]]

    df = df.merge(old, on=key, how="left", suffixes=("", "_old"))

    for c in all_output_cols():
        oldc = f"{c}_old"
        if oldc in df.columns:
            df[c] = df[c].where(df[c].astype(str).str.strip() != "", df[oldc].fillna(""))
            df.drop(columns=[oldc], inplace=True)

    return df

# =========================
# Inference
# =========================
async def call_model(model_id: str, user_msg: str) -> str:
    # GPT-5 family -> Responses API
    if model_id.startswith("gpt-5"):
        resp = await client.responses.create(
            model=model_id,
            input=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            reasoning={"effort": "none"},   # or "low"
            text={"verbosity": "low"},
            max_output_tokens=4000,         # start here; increase if needed
        )
        return (resp.output_text or "").strip()

    # GPT-4.1 family -> Chat Completions API
    else:
        resp = await client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            temperature=TEMPERATURE,
            max_tokens=800,
        )
        return (resp.choices[0].message.content or "").strip()
    

async def score_dw_with_reason(
    model_name: str,
    row: pd.Series,
    sema: asyncio.Semaphore,
    meta: str,
    row_idx: int,
) -> Dict[str, Any]:
    async with sema:
        user_msg = build_user_message(row)
        model_id = MODEL_CFG[model_name]["model"]

        async def _primary():
            return await call_model(model_id, user_msg)

        res = await with_retries(_primary, meta=meta)
        if isinstance(res, str) and res.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "api_error")
            return {"score": None, "reason": res}

        obj = extract_json_from_text(res) or code_repair_json(res)
        if obj is not None:
            ok, _, s, r = validate_dw_reason(obj)
            if ok:
                RUN_STATS[model_name]["ok"] += 1
                return {"score": s, "reason": r}

        if not ENABLE_REPAIR_CALL:
            RUN_STATS[model_name]["failed"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "failed", note="No repair call enabled.")
            return {"score": None, "reason": "INVALID_JSON_NO_REPAIR"}

        obj_pre = extract_json_from_text(res) or code_repair_json(res)
        fail_reason = validate_dw_reason(obj_pre)[1] if obj_pre is not None else "unparseable JSON"

        repair_system = (
        "Fix the following output into VALID JSON ONLY.\n"
        'Required format: {"score": 3, "reason": "3-5 sentences with a direct quote fragment."}\n'
        "No markdown. No extra text. No extra keys."
    )
        repair_user = (
            f"Your previous output failed validation. Reason: {fail_reason}\n\n"
            f"Broken output:\n{res}"
        )

        async def _repair():
            resp = await client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": repair_system},
                    {"role": "user",   "content": repair_user},
                ],
                temperature=0,
                max_tokens=MAX_TOKENS,
            )
            return (resp.choices[0].message.content or "").strip()

        res2 = await with_retries(_repair, meta=meta + "|repair")
        if isinstance(res2, str) and res2.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res2, "api_error", broken_output=res)
            return {"score": None, "reason": res2}

        obj2 = extract_json_from_text(res2) or code_repair_json(res2)
        if obj2 is not None:
            ok2, _, s2, r2 = validate_dw_reason(obj2)
            if ok2:
                RUN_STATS[model_name]["repaired"] += 1
                save_inference_log(
                    model_name, row_idx, user_msg, res2, "repaired",
                    broken_output=res,
                    note="Primary response failed validation and was repaired."
                )
                return {"score": s2, "reason": r2}

        RUN_STATS[model_name]["failed"] += 1
        save_inference_log(
            model_name, row_idx, user_msg, res, "failed",
            note="Failed after repair attempt."
        )
        return {"score": None, "reason": "INVALID_JSON_AFTER_REPAIR"}

# =========================
# Sequential runner
# =========================
async def run_one_model(
    df: pd.DataFrame,
    model_name: str,
    sema: asyncio.Semaphore,
    process_indices: List[int],
) -> None:
    cfg = MODEL_CFG[model_name]
    score_col  = cfg["score_col"]
    reason_col = cfg["reason_col"]

    for k in range(0, len(process_indices), BATCH_SIZE):
        batch = process_indices[k:k + BATCH_SIZE]
        todo  = [i for i in batch if not row_done(df, i, model_name)]
        if not todo:
            continue

        async def run_row(ii: int):
            meta = f"{model_name}|row={ii}"
            obj = await score_dw_with_reason(model_name, df.iloc[ii], sema, meta, ii)
            return ii, obj

        tasks = [asyncio.create_task(run_row(i)) for i in todo]

        batch_num = k // BATCH_SIZE + 1
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"{model_name} batch {batch_num}"):
            i, obj = await fut
            score = obj.get("score")
            reason = obj.get("reason")

            df.at[i, score_col]  = "" if score is None else str(score)
            df.at[i, reason_col] = "" if reason is None else str(reason)

        save_full(df, OUTPUT_PATH)
        print(f"[Checkpoint] Saved → {OUTPUT_PATH}")

# =========================
# Summary
# =========================
def print_summary():
    print("\n================ SUMMARY ================")
    for model_name, cfg in enabled_models().items():
        stats = RUN_STATS[model_name]
        ok = stats.get("ok", 0)
        repaired = stats.get("repaired", 0)
        failed = stats.get("failed", 0)
        api_error = stats.get("api_error", 0)
        total_processed = ok + repaired + failed + api_error

        print(f"{model_name}:")
        print(f"  model       = {cfg['model']}")
        print(f"  processed   = {total_processed}")
        print(f"  ok          = {ok}")
        print(f"  repaired    = {repaired}")
        print(f"  failed      = {failed}")
        print(f"  api_error   = {api_error}")
        print(f"  usable      = {ok + repaired}")
        print()

    print(f"Log folder: {RUN_LOG_DIR}")
    print("=========================================\n")

# =========================
# MAIN
# =========================
async def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(INPUT_FILE)

    global OUTPUT_PATH
    OUTPUT_PATH = safe_test_output_path(OUTPUT_FILE) if DEBUG_TEST_ONLY else OUTPUT_FILE

    df = _read_tabular(INPUT_FILE)
    df = ensure_cols(df)

    if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_PATH):
        df = resume_merge(df, OUTPUT_PATH)

    for c in ["paragraph_clean", "party"]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    active_models = enabled_models()
    if not active_models:
        raise ValueError("No models enabled in MODEL_CFG.")

    n = len(df)
    process_indices = list(range(min(DEBUG_N_TEST, n))) if DEBUG_TEST_ONLY else list(range(n))
    sema = asyncio.Semaphore(CONCURRENCY)

    print(f"Rows: {n:,} | Processing: {len(process_indices):,}")
    print(f"Output: {OUTPUT_PATH}")
    print(f"Logs root: {TEMP_DIR}")
    print(f"Run logs:  {RUN_LOG_DIR}")
    print(f"Test mode: {DEBUG_TEST_ONLY}")
    print(f"Enabled models: {list(active_models.keys())}")

    for model_name, cfg in active_models.items():
        print(f"\n=== Running {model_name} sequentially ===")
        print(cfg["model"])
        await run_one_model(df, model_name, sema, process_indices)

    save_full(df, OUTPUT_PATH)
    print_summary()
    print("✅ DONE")

# RUN
await main()

Rows: 551 | Processing: 551
Output: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx
Logs root: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/temp/_openai_inference_logs
Run logs:  /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/temp/_openai_inference_logs/run_20260317T233315
Test mode: False
Enabled models: ['gpt41_fs', 'gpt5_mini_fs', 'gpt5_fs']

=== Running gpt41_fs sequentially ===
gpt-4.1-2025-04-14


gpt41_fs batch 1: 100%|██████████| 80/80 [02:19<00:00,  1.74s/it]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx


gpt41_fs batch 2: 100%|██████████| 80/80 [02:10<00:00,  1.63s/it]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx


...

[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx


gpt5_fs batch 6: 100%|██████████| 80/80 [02:01<00:00,  1.51s/it]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx


gpt5_fs batch 7: 100%|██████████| 71/71 [01:54<00:00,  1.61s/it]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_finetuned_dw_multi_example_models.xlsx

================ SUMMARY ================
gpt41_fs:
  model       = gpt-4.1-2025-04-14
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

gpt5_mini_fs:
  model       = gpt-5.4-mini-2026-03-17
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

gpt5_fs:
  model       = gpt-5.4-2026-03-05
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

Log folder: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/temp/_openai_inference_logs/run_20260317T233315

✅ DONE


### RAG
This cell builds a **RAG-ready fine-tuning dataset** from the stratified DW speech sample by combining embedding-based retrieval with JSONL export for multiple training formats.

Steps:

1. **Load and clean** the source Excel file (`speech_sample_200_stratified.xlsx`).
2. **Normalize fields** (`party`, `paragraph_clean`, `share_yes`) and remove unusable rows.
3. **Create ordinal labels** (`human_score_ferenc`, 1–5) from `share_yes`.
4. **Split data** into train/validation (80/20), stratified by score.
5. **Generate embeddings** for train and validation passages using `text-embedding-3-large`.
6. **Retrieve top-K neighbors** for each example using cosine similarity:
    - train queries exclude self-match
    - validation queries search the train embedding index
7. **Export retrieval diagnostics** (neighbors, similarity, source/neighbor metadata) to wrapped Excel files.
8. **Build RAG prompts** that include K similar labeled examples for each target passage.
9. **Write supervised JSONL datasets** for three ecosystems:
    - OpenAI
    - Gemini/Vertex
    - Open-model chat format
10. **Save artifacts** for reproducible training and inspection.

Configuration:

- `BASE_DIR` / `EXCEL_PATH` : source dataset location  
- `EMBED_DIR` : embedding and retrieval artifacts  
- `OUTPUT_DIR` : final RAG JSONL files  
- `EMBED_MODEL` : embedding model (`text-embedding-3-large`)  
- `VAL_SIZE` / `RANDOM_STATE` : split settings  
- `BATCH_SIZE` : embedding API batch size  
- `K` : number of retrieved neighbors per example

Outputs:

- Embedding artifacts (`embeddings.npy`, `embed_meta.jsonl`)
- Split tracking files (`train_split_for_rag.xlsx`, `val_split_for_rag.xlsx`, `full_with_split_flags.xlsx`)
- Neighbor inspection files (`neighbors_train.xlsx`, `neighbors_val.xlsx`)
- RAG training/validation JSONL files for OpenAI, Gemini, and Open formats

In [6]:
"""
build_embeddings.py

1. Embeds train-split paragraphs (text-embedding-3-large)
2. Retrieves K nearest neighbors for every train + val example
3. Saves neighbor Excels with similarity scores + wrapped text
4. Builds RAG-augmented JSONL files (OpenAI, Gemini, Open)

Outputs in EMBED_DIR  (BASE_DIR/temp/rag_embeddings_dw_collusion_sample200):
  embeddings.npy
  embed_meta.jsonl
  train_split_for_rag.xlsx
  val_split_for_rag.xlsx
  full_with_split_flags.xlsx
  neighbors_train.xlsx
  neighbors_val.xlsx

Outputs in OUTPUT_DIR (BASE_DIR/temp/rag_jsonl_dw_collusion_sample200):
  train_openai_rag.jsonl / val_openai_rag.jsonl
  train_gemini_rag.jsonl / val_gemini_rag.jsonl
  train_open_rag.jsonl   / val_open_rag.jsonl
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from openai import OpenAI
import openpyxl
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

# =========================
# CONFIG
# =========================
BASE_DIR   = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
EXCEL_PATH = os.path.join(BASE_DIR, "input", "ft_sample", "speech_sample_200_stratified.xlsx")

EMBED_DIR  = os.path.join(BASE_DIR, "temp", "rag_embeddings_dw_collusion_sample200")
OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", "rag_jsonl_dw_collusion_sample200")

os.makedirs(EMBED_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL  = "text-embedding-3-large"   # best OpenAI embedding model
RANDOM_STATE = 42
VAL_SIZE     = 0.20
BATCH_SIZE   = 50
K            = 3   # neighbors per example

with open("input/api's/openai_api.txt", "r") as f:
    api_key = f.read().strip()

# =========================
# LOAD + PREPROCESS
# =========================
df = pd.read_excel(EXCEL_PATH).copy()
df["orig_row_id"] = np.arange(len(df))

df["party"]           = df["party"].replace({"Democrat": "Democratic"}).fillna("Unknown")
df["paragraph_clean"] = df["paragraph_clean"].astype(str).str.strip()
df["share_yes"]       = pd.to_numeric(df["share_yes"], errors="coerce")
df["share_yes"]       = np.where(df["share_yes"] > 1, df["share_yes"] / 100, df["share_yes"])
df["share_yes"]       = df["share_yes"].clip(0, 1)

df = df.dropna(subset=["share_yes"])
df = df[df["paragraph_clean"].ne("")].copy().reset_index(drop=True)

if "q_id" not in df.columns:
    df["q_id"] = [f"row_{i:04d}" for i in range(len(df))]

bins   = [-0.001, 0.20, 0.40, 0.60, 0.80, 1.001]
labels = [1, 2, 3, 4, 5]
df["human_score_ferenc"] = pd.cut(
    df["share_yes"], bins=bins, labels=labels, include_lowest=True, right=False
).astype(int)

# =========================
# SPLIT
# =========================
train_idx, val_idx = train_test_split(
    range(len(df)),
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["human_score_ferenc"]
)
train_idx = sorted(train_idx)
val_idx   = sorted(val_idx)

df["split"] = "unused"
df.loc[train_idx, "split"] = "train"
df.loc[val_idx,   "split"] = "val"

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}")

train_df.to_excel(os.path.join(EMBED_DIR, "train_split_for_rag.xlsx"), index=False)
val_df.to_excel(  os.path.join(EMBED_DIR, "val_split_for_rag.xlsx"),   index=False)
df.to_excel(      os.path.join(EMBED_DIR, "full_with_split_flags.xlsx"), index=False)
print("Saved split Excel files.")

# =========================
# HELPER: SAVE EXCEL WITH WRAPPED TEXT
# =========================
# Column widths: key = column name substring, value = width
COLUMN_WIDTHS = {
    "paragraph": 80,
    "q_id":      14,
    "party":     14,
    "score":     10,
    "rank":      8,
    "sim":       12,
}

def save_excel_wrapped(df_out, path):
    """Save a DataFrame to Excel with text wrapping and sensible column widths."""
    df_out.to_excel(path, index=False, engine="openpyxl")
    wb = openpyxl.load_workbook(path)
    ws = wb.active

    for col_idx, col_name in enumerate(df_out.columns, start=1):
        col_letter = get_column_letter(col_idx)

        # pick width
        width = 18   # default
        for key, w in COLUMN_WIDTHS.items():
            if key in col_name.lower():
                width = w
                break
        ws.column_dimensions[col_letter].width = width

        # wrap every cell in this column (including header)
        for cell in ws[col_letter]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    # freeze header row
    ws.freeze_panes = "A2"
    wb.save(path)

# =========================
# EMBED
# =========================
client = OpenAI(api_key=api_key)

def embed_texts_in_batches(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE):
    all_embeddings = []
    n = len(texts)
    for start in range(0, n, batch_size):
        end      = min(start + batch_size, n)
        response = client.embeddings.create(model=model, input=texts[start:end])
        batch_embs = [None] * len(response.data)
        for item in response.data:
            batch_embs[item.index] = item.embedding
        all_embeddings.extend(batch_embs)
        print(f"  Embedded {end}/{n}")
    return np.array(all_embeddings, dtype=np.float32)

def normalize(vecs):
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / np.clip(norms, 1e-9, None)

print("Embedding train set...")
train_embeds = embed_texts_in_batches(train_df["paragraph_clean"].tolist())
train_normed = normalize(train_embeds)
print(f"Train embedding shape: {train_embeds.shape}")

np.save(os.path.join(EMBED_DIR, "embeddings.npy"), train_embeds)

with open(os.path.join(EMBED_DIR, "embed_meta.jsonl"), "w", encoding="utf-8") as f:
    for i, row in train_df.iterrows():
        f.write(json.dumps({
            "idx":             i,
            "q_id":            row["q_id"],
            "orig_row_id":     int(row["orig_row_id"]),
            "paragraph_clean": row["paragraph_clean"],
            "party":           row["party"],
            "share_yes":       round(float(row["share_yes"]), 4),
            "score":           int(row["human_score_ferenc"])
        }, ensure_ascii=False) + "\n")
print("Saved embeddings.npy and embed_meta.jsonl.")

print("Embedding val set...")
val_normed = normalize(embed_texts_in_batches(val_df["paragraph_clean"].tolist()))

# =========================
# RETRIEVE K NEIGHBORS
# Returns list of lists of (train_idx, similarity_score) tuples
# =========================
def retrieve_neighbors(query_normed, k, exclude_self=False):
    sims    = query_normed @ train_normed.T   # (N_query, N_train)
    results = []
    for i, sim_row in enumerate(sims):
        if exclude_self:
            sim_row    = sim_row.copy()
            sim_row[i] = -1.0
        top_k = np.argsort(sim_row)[::-1][:k]
        results.append([(int(ni), float(sim_row[ni])) for ni in top_k])
    return results

train_neighbors = retrieve_neighbors(train_normed, K, exclude_self=True)
val_neighbors   = retrieve_neighbors(val_normed,   K, exclude_self=False)

# =========================
# SAVE NEIGHBORS EXCEL (with similarity + wrapped text)
# =========================
def build_neighbors_df(source_df, neighbors):
    rows = []
    for i, (_, src) in enumerate(source_df.iterrows()):
        for rank, (ni, sim_score) in enumerate(neighbors[i], start=1):
            nb = train_df.iloc[ni]
            rows.append({
                "source_q_id":        src["q_id"],
                "source_paragraph":   src["paragraph_clean"],
                "source_party":       src["party"],
                "source_score":       int(src["human_score_ferenc"]),
                "rank":               rank,
                "similarity":         round(sim_score, 4),
                "neighbor_q_id":      nb["q_id"],
                "neighbor_paragraph": nb["paragraph_clean"],
                "neighbor_party":     nb["party"],
                "neighbor_score":     int(nb["human_score_ferenc"]),
            })
    return pd.DataFrame(rows)

neighbors_train_df = build_neighbors_df(train_df, train_neighbors)
neighbors_val_df   = build_neighbors_df(val_df,   val_neighbors)

save_excel_wrapped(neighbors_train_df, os.path.join(EMBED_DIR, "neighbors_train.xlsx"))
save_excel_wrapped(neighbors_val_df,   os.path.join(EMBED_DIR, "neighbors_val.xlsx"))
print("Saved neighbors Excel files.")

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{"score": <integer between 1 and 5 inclusive>}
""".strip()

def build_user_msg(party, passage, neighbors):
    examples = "\n\n".join([
        f'Example {r+1} ({train_df.iloc[ni]["party"]} party, score {int(train_df.iloc[ni]["human_score_ferenc"])}, similarity {sim:.3f}):\n'
        f'"{train_df.iloc[ni]["paragraph_clean"]}"'
        for r, (ni, sim) in enumerate(neighbors)
    ])
    return (
        f"Here are {K} similar passages and their human scores:\n\n"
        f"{examples}\n\n"
        f"---\n\n"
        f"The speaker is affiliated with the {party} party.\n"
        f"If the paragraph quotes someone from a different political party, "
        f"the speaker might disagree with the quoted content.\n\n"
        f'Passage:\n"{passage}"'
    )

# =========================
# BUILD + WRITE JSONL
# =========================
def build_and_write(split_df, neighbors, prefix):
    openai_recs, gemini_recs, open_recs = [], [], []
    for i, (_, row) in enumerate(split_df.iterrows()):
        user_msg = build_user_msg(row["party"], row["paragraph_clean"], neighbors[i])
        target   = json.dumps({"score": int(row["human_score_ferenc"])}, ensure_ascii=False)

        openai_recs.append({"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]})
        gemini_recs.append({
            "systemInstruction": {"role": "system", "parts": [{"text": SYSTEM_PROMPT}]},
            "contents": [
                {"role": "user",  "parts": [{"text": user_msg}]},
                {"role": "model", "parts": [{"text": target}]}
            ]
        })
        open_recs.append({"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]})

    def write_jsonl(path, data):
        with open(path, "w", encoding="utf-8") as f:
            for rec in data:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print("Saved:", path)

    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_openai_rag.jsonl"), openai_recs)
    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_gemini_rag.jsonl"), gemini_recs)
    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_open_rag.jsonl"),   open_recs)

build_and_write(train_df, train_neighbors, "train")
build_and_write(val_df,   val_neighbors,   "val")

print(f"\nDone. Model={EMBED_MODEL}  K={K} neighbors per example.")
print(f"Embeddings -> {EMBED_DIR}")
print(f"JSONL      -> {OUTPUT_DIR}")

Train: 160  Val: 40
Saved split Excel files.
Embedding train set...
  Embedded 50/160
  Embedded 100/160
  Embedded 150/160
  Embedded 160/160
Train embedding shape: (160, 3072)
Saved embeddings.npy and embed_meta.jsonl.
Embedding val set...
  Embedded 40/40
Saved neighbors Excel files.
Saved: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/ft_sample_jason/rag_jsonl_dw_collusion_sample200/train_openai_rag.jsonl
Saved: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/ft_sample_jason/rag_jsonl_dw_collusion_sample200/train_gemini_rag.jsonl
Saved: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/ft_sample_jason/rag_jsonl_dw_collusion_sample200/train_open_rag.jsonl
Saved: /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/ft_sample_jason/rag_jsonl_dw_collusion_sample200/val_openai_rag.jsonl
Saved: /Users/ghost/Libr

In [7]:
"""
OpenAI Fine-Tune Launcher (no monitoring, notebook-friendly)
===========================================================
- Auto-resumes from job log if interrupted
- Launches a new job if the previous one is terminal
- No polling / no progress bar / no SystemExit
"""
# ── CONFIG ─────────────────────────────────────────────────────────────────────

DATASET_FOLDER = "rag_jsonl_dw_collusion_sample200"

OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", DATASET_FOLDER)

TRAIN_PATH = os.path.join(OUTPUT_DIR, "train_openai_rag.jsonl")
VAL_PATH   = os.path.join(OUTPUT_DIR, "val_openai_rag.jsonl")

JOB_LOG = os.path.join(OUTPUT_DIR, "finetune_job_log_openai_rag.json")

MODEL = "gpt-4.1-2025-04-14"

# Hyperparameters
N_EPOCHS   = 3
BATCH_SIZE = 4
LR_MULT    = 0.5

# Model suffix (important for distinguishing experiments)
SUFFIX = "dw_collusion_ordinal_rag200"


# ── HELPERS ────────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def load_log():
    try:
        return json.load(open(JOB_LOG, encoding="utf-8")) if os.path.exists(JOB_LOG) else {}
    except Exception:
        return {}

def save_log(data):
    os.makedirs(os.path.dirname(JOB_LOG), exist_ok=True)
    with open(JOB_LOG, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)


# ── LAUNCH OR RESUME ───────────────────────────────────────────────────────────

job_log = load_log()
job_id  = job_log.get("job_id")

if job_id:
    existing = client.fine_tuning.jobs.retrieve(job_id)

    if existing.status in ("succeeded", "failed", "cancelled"):
        log(f"Previous job {job_id} already {existing.status} — launching new job.")
        job_id = None

    else:
        log(f"Resuming existing job {job_id} (status={existing.status})")
        log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

        save_log({
            **job_log,
            "status": existing.status,
            "resumed_at": datetime.now().isoformat()
        })

        job_id = existing.id


# ── START NEW JOB ──────────────────────────────────────────────────────────────

if not job_id:

    train_n = count_lines(TRAIN_PATH)
    val_n   = count_lines(VAL_PATH)

    log(f"Train rows: {train_n}  |  Val rows: {val_n}")

    log("Uploading files...")

    with open(TRAIN_PATH, "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")

    with open(VAL_PATH, "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")

    log(f"Train file id: {train_file.id}")
    log(f"Val file id:   {val_file.id}")

    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=MODEL,
        hyperparameters={
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate_multiplier": LR_MULT
        },
        suffix=SUFFIX
    )

    job_id = job.id

    log(f"✅ Job launched: {job_id}")
    log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

    save_log({
        "job_id": job_id,
        "model": MODEL,
        "train_rows": train_n,
        "val_rows": val_n,
        "train_file_id": train_file.id,
        "val_file_id": val_file.id,
        "hyperparameters": {
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr_mult": LR_MULT
        },
        "suffix": SUFFIX,
        "launched_at": datetime.now().isoformat(),
        "status": job.status
    })


# ── CHECK INITIAL STATUS ───────────────────────────────────────────────────────

latest = client.fine_tuning.jobs.retrieve(job_id)

log(f"Initial status: {latest.status}")
log("Done. You can run other code now.")

[00:38:20] Train rows: 160  |  Val rows: 40
[00:38:20] Uploading files...
[00:38:22] Train file id: file-Bs6oknDDg7G56jtFFQ696A
[00:38:22] Val file id:   file-7rzpvpJFtenPe5YGpx5MXM
[00:38:25] ✅ Job launched: ftjob-wrY9joDaQvygTvoCIqceHvIJ
[00:38:25] OpenAI UI: https://platform.openai.com/finetune/ftjob-wrY9joDaQvygTvoCIqceHvIJ
[00:38:25] Initial status: validating_files
[00:38:25] Done. You can run other code now.


#### RFF scoring

In [8]:
"""
build_embeddings_hybrid.py

1. Embeds train-split paragraphs (text-embedding-3-large)
2. Retrieves K nearest neighbors using Hybrid RRF (cosine + BM25)
3. Saves neighbor Excels with RRF scores + wrapped text
4. Builds RAG-augmented JSONL files (OpenAI, Gemini, Open)

Outputs in EMBED_DIR  (BASE_DIR/temp/rag_embeddings_hybrid_dw_collusion_sample300):
  embeddings.npy
  embed_meta.jsonl
  train_split_for_rag.xlsx
  val_split_for_rag.xlsx
  full_with_split_flags.xlsx
  neighbors_train.xlsx
  neighbors_val.xlsx

Outputs in OUTPUT_DIR (BASE_DIR/output/ft_sample_jason/rag_jsonl_hybrid_dw_collusion_sample300):
  train_openai_rag.jsonl / val_openai_rag.jsonl
  train_gemini_rag.jsonl / val_gemini_rag.jsonl
  train_open_rag.jsonl   / val_open_rag.jsonl
"""
# =========================
# CONFIG
# =========================
#BASE_DIR   = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
EXCEL_PATH = os.path.join(BASE_DIR, "input", "ft_sample", "speech_sample_300_stratified.xlsx")

EMBED_DIR  = os.path.join(BASE_DIR, "temp",   "rag_embeddings_hybrid_dw_collusion_sample300")
OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", "rag_jsonl_hybrid_dw_collusion_sample300")

os.makedirs(EMBED_DIR,  exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMBED_MODEL  = "text-embedding-3-large"
RANDOM_STATE = 42
VAL_SIZE     = 0.20
BATCH_SIZE   = 50
K            = 3     # neighbors per example
RRF_K        = 60    # standard RRF constant

with open("input/api's/openai_api.txt", "r") as f:
    api_key = f.read().strip()

# =========================
# LOAD + PREPROCESS
# =========================
df = pd.read_excel(EXCEL_PATH).copy()
df["orig_row_id"] = np.arange(len(df))

df["party"]           = df["party"].replace({"Democrat": "Democratic"}).fillna("Unknown")
df["paragraph_clean"] = df["paragraph_clean"].astype(str).str.strip()
df["share_yes"]       = pd.to_numeric(df["share_yes"], errors="coerce")
df["share_yes"]       = np.where(df["share_yes"] > 1, df["share_yes"] / 100, df["share_yes"])
df["share_yes"]       = df["share_yes"].clip(0, 1)

df = df.dropna(subset=["share_yes"])
df = df[df["paragraph_clean"].ne("")].copy().reset_index(drop=True)

if "q_id" not in df.columns:
    df["q_id"] = [f"row_{i:04d}" for i in range(len(df))]

bins   = [-0.001, 0.20, 0.40, 0.60, 0.80, 1.001]
labels = [1, 2, 3, 4, 5]
df["human_score_ferenc"] = pd.cut(
    df["share_yes"], bins=bins, labels=labels, include_lowest=True, right=False
).astype(int)

# =========================
# SPLIT
# =========================
train_idx, val_idx = train_test_split(
    range(len(df)),
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["human_score_ferenc"]
)
train_idx = sorted(train_idx)
val_idx   = sorted(val_idx)

df["split"] = "unused"
df.loc[train_idx, "split"] = "train"
df.loc[val_idx,   "split"] = "val"

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}")

train_df.to_excel(os.path.join(EMBED_DIR, "train_split_for_rag.xlsx"), index=False)
val_df.to_excel(  os.path.join(EMBED_DIR, "val_split_for_rag.xlsx"),   index=False)
df.to_excel(      os.path.join(EMBED_DIR, "full_with_split_flags.xlsx"), index=False)
print("Saved split Excel files.")

# =========================
# HELPER: SAVE EXCEL WITH WRAPPED TEXT
# =========================
COLUMN_WIDTHS = {
    "paragraph": 80,
    "q_id":      14,
    "party":     14,
    "score":     10,
    "share_yes": 12,
    "rank":      8,
    "rrf":       12,
}

def save_excel_wrapped(df_out, path):
    df_out.to_excel(path, index=False, engine="openpyxl")
    wb = openpyxl.load_workbook(path)
    ws = wb.active

    for col_idx, col_name in enumerate(df_out.columns, start=1):
        col_letter = get_column_letter(col_idx)
        width = 18
        for key, w in COLUMN_WIDTHS.items():
            if key in col_name.lower():
                width = w
                break
        ws.column_dimensions[col_letter].width = width
        for cell in ws[col_letter]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

    ws.freeze_panes = "A2"
    wb.save(path)

# =========================
# EMBED
# =========================
client = OpenAI(api_key=api_key)

def embed_texts_in_batches(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE):
    all_embeddings = []
    n = len(texts)
    for start in range(0, n, batch_size):
        end      = min(start + batch_size, n)
        response = client.embeddings.create(model=model, input=texts[start:end])
        batch_embs = [None] * len(response.data)
        for item in response.data:
            batch_embs[item.index] = item.embedding
        all_embeddings.extend(batch_embs)
        print(f"  Embedded {end}/{n}")
    return np.array(all_embeddings, dtype=np.float32)

def normalize(vecs):
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / np.clip(norms, 1e-9, None)

print("Embedding train set...")
train_embeds = embed_texts_in_batches(train_df["paragraph_clean"].tolist())
train_normed = normalize(train_embeds)
print(f"Train embedding shape: {train_embeds.shape}")

np.save(os.path.join(EMBED_DIR, "embeddings.npy"), train_embeds)

with open(os.path.join(EMBED_DIR, "embed_meta.jsonl"), "w", encoding="utf-8") as f:
    for i, row in train_df.iterrows():
        f.write(json.dumps({
            "idx":             i,
            "q_id":            row["q_id"],
            "orig_row_id":     int(row["orig_row_id"]),
            "paragraph_clean": row["paragraph_clean"],
            "party":           row["party"],
            "share_yes":       round(float(row["share_yes"]), 4),
            "score":           int(row["human_score_ferenc"])
        }, ensure_ascii=False) + "\n")
print("Saved embeddings.npy and embed_meta.jsonl.")

print("Embedding val set...")
val_normed = normalize(embed_texts_in_batches(val_df["paragraph_clean"].tolist()))

# =========================
# BM25 INDEX  (train corpus only)
# FIX: regex tokenizer strips punctuation so "collusion," == "collusion"
# =========================
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

train_corpus = train_df["paragraph_clean"].tolist()
bm25_index   = BM25Okapi([tokenize(t) for t in train_corpus])
print("BM25 index built.")

# =========================
# HYBRID RETRIEVE  (RRF: cosine rank + BM25 rank)
# FIX: +1 to convert 0-indexed ranks to 1-indexed before RRF
# Returns list of lists of (train_idx, rrf_score) tuples, length k each
# =========================
def retrieve_neighbors_hybrid(query_normed, query_texts, k, exclude_self=False):
    cos_sims = query_normed @ train_normed.T   # (N_query, N_train)
    results  = []

    for i, query_text in enumerate(query_texts):
        cos_row     = cos_sims[i].copy()
        bm25_scores = np.array(bm25_index.get_scores(tokenize(query_text)))

        if exclude_self:
            cos_row[i]     = -1.0
            bm25_scores[i] = -1.0

        # 0-indexed ranks → add 1 for correct 1-indexed RRF
        cos_ranks  = np.argsort(np.argsort(-cos_row))  + 1
        bm25_ranks = np.argsort(np.argsort(-bm25_scores)) + 1

        # RRF fusion
        rrf_scores = 1.0 / (RRF_K + cos_ranks) + 1.0 / (RRF_K + bm25_ranks)

        top_k = np.argsort(rrf_scores)[::-1][:k]
        results.append([(int(ni), float(rrf_scores[ni])) for ni in top_k])

    return results

print("Retrieving neighbors (hybrid RRF)...")
train_neighbors = retrieve_neighbors_hybrid(
    train_normed, train_df["paragraph_clean"].tolist(), K, exclude_self=True
)
val_neighbors = retrieve_neighbors_hybrid(
    val_normed, val_df["paragraph_clean"].tolist(), K, exclude_self=False
)

# =========================
# SAVE NEIGHBORS EXCEL
# FIX: include share_yes for both source and neighbor
# =========================
def build_neighbors_df(source_df, neighbors):
    rows = []
    for i, (_, src) in enumerate(source_df.iterrows()):
        for rank, (ni, rrf_score) in enumerate(neighbors[i], start=1):
            nb = train_df.iloc[ni]
            rows.append({
                "source_q_id":        src["q_id"],
                "source_paragraph":   src["paragraph_clean"],
                "source_party":       src["party"],
                "source_share_yes":   round(float(src["share_yes"]), 4),
                "source_score":       int(src["human_score_ferenc"]),
                "rank":               rank,
                "rrf_score":          round(rrf_score, 6),
                "neighbor_q_id":      nb["q_id"],
                "neighbor_paragraph": nb["paragraph_clean"],
                "neighbor_party":     nb["party"],
                "neighbor_share_yes": round(float(nb["share_yes"]), 4),
                "neighbor_score":     int(nb["human_score_ferenc"]),
            })
    return pd.DataFrame(rows)

save_excel_wrapped(
    build_neighbors_df(train_df, train_neighbors),
    os.path.join(EMBED_DIR, "neighbors_train.xlsx")
)
save_excel_wrapped(
    build_neighbors_df(val_df, val_neighbors),
    os.path.join(EMBED_DIR, "neighbors_val.xlsx")
)
print("Saved neighbors Excel files.")

# =========================
# PROMPTS
# FIX: rrf_score removed from user message — internal metric, not useful to model
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{"score": <integer between 1 and 5 inclusive>}
""".strip()

def build_user_msg(party, passage, neighbors):
    examples = "\n\n".join([
        f'Example {r+1} ({train_df.iloc[ni]["party"]} party, score {int(train_df.iloc[ni]["human_score_ferenc"])}):\n'
        f'"{train_df.iloc[ni]["paragraph_clean"]}"'
        for r, (ni, _) in enumerate(neighbors)
    ])
    return (
        f"Here are {K} similar passages and their human scores:\n\n"
        f"{examples}\n\n"
        f"---\n\n"
        f"The speaker is affiliated with the {party} party.\n"
        f"If the paragraph quotes someone from a different political party, "
        f"the speaker might disagree with the quoted content.\n\n"
        f'Passage:\n"{passage}"'
    )

# =========================
# BUILD + WRITE JSONL
# =========================
def build_and_write(split_df, neighbors, prefix):
    openai_recs, gemini_recs, open_recs = [], [], []
    for i, (_, row) in enumerate(split_df.iterrows()):
        user_msg = build_user_msg(row["party"], row["paragraph_clean"], neighbors[i])
        target   = json.dumps({"score": int(row["human_score_ferenc"])}, ensure_ascii=False)

        openai_recs.append({"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]})
        gemini_recs.append({
            "systemInstruction": {"role": "system", "parts": [{"text": SYSTEM_PROMPT}]},
            "contents": [
                {"role": "user",  "parts": [{"text": user_msg}]},
                {"role": "model", "parts": [{"text": target}]}
            ]
        })
        open_recs.append({"messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": target}
        ]})

    def write_jsonl(path, data):
        with open(path, "w", encoding="utf-8") as f:
            for rec in data:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print("Saved:", path)

    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_openai_rag.jsonl"), openai_recs)
    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_gemini_rag.jsonl"), gemini_recs)
    write_jsonl(os.path.join(OUTPUT_DIR, f"{prefix}_open_rag.jsonl"),   open_recs)

build_and_write(train_df, train_neighbors, "train")
build_and_write(val_df,   val_neighbors,   "val")

print(f"\nDone. Model={EMBED_MODEL}  K={K}  RRF_K={RRF_K}")
print(f"Embeddings -> {EMBED_DIR}")
print(f"JSONL      -> {OUTPUT_DIR}")

Train: 240  Val: 60
Saved split Excel files.
Embedding train set...
  Embedded 50/240
  Embedded 100/240
  Embedded 150/240
  Embedded 200/240
  Embedded 240/240
Train embedding shape: (240, 3072)
Saved embeddings.npy and embed_meta.jsonl.
Embedding val set...
  Embedded 50/60
  Embedded 60/60
BM25 index built.
Retrieving neighbors (hybrid RRF)...
Saved neighbors Excel files.
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\ft_sample_jason\rag_jsonl_hybrid_dw_collusion_sample300\train_openai_rag.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\ft_sample_jason\rag_jsonl_hybrid_dw_collusion_sample300\train_gemini_rag.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\ft_sample_jason\rag_jsonl_hybrid_dw_collusion_sample300\train_open_rag.jsonl
Saved: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\ft_sample_jason\rag_jsonl_hybrid_dw_collusion_

In [11]:
"""
OpenAI Fine-Tune Launcher (no monitoring, notebook-friendly)
===========================================================
- Auto-resumes from job log if interrupted
- Launches a new job if the previous one is terminal
- No polling / no progress bar / no SystemExit
"""
# ── CONFIG ─────────────────────────────────────────────────────────────────────

DATASET_FOLDER = "rag_jsonl_hybrid_dw_collusion_sample300"

OUTPUT_DIR = os.path.join(BASE_DIR, "output", "ft_sample_jason", DATASET_FOLDER)

TRAIN_PATH = os.path.join(OUTPUT_DIR, "train_openai_rag.jsonl")
VAL_PATH   = os.path.join(OUTPUT_DIR, "val_openai_rag.jsonl")

JOB_LOG = os.path.join(OUTPUT_DIR, "finetune_job_log_openai_rag_hybrid.json")

MODEL = "gpt-4.1-2025-04-14"

# Hyperparameters
N_EPOCHS   = 3
BATCH_SIZE = 4
LR_MULT    = 0.5

# Model suffix (important for distinguishing experiments)
SUFFIX = "dw_collusion_ordinal_rag_hybrid300"


# ── HELPERS ────────────────────────────────────────────────────────────────────

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

def load_log():
    try:
        return json.load(open(JOB_LOG, encoding="utf-8")) if os.path.exists(JOB_LOG) else {}
    except Exception:
        return {}

def save_log(data):
    os.makedirs(os.path.dirname(JOB_LOG), exist_ok=True)
    with open(JOB_LOG, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def count_lines(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for _ in f)


# ── LAUNCH OR RESUME ───────────────────────────────────────────────────────────

job_log = load_log()
job_id  = job_log.get("job_id")

if job_id:
    existing = client.fine_tuning.jobs.retrieve(job_id)

    if existing.status in ("succeeded", "failed", "cancelled"):
        log(f"Previous job {job_id} already {existing.status} — launching new job.")
        job_id = None

    else:
        log(f"Resuming existing job {job_id} (status={existing.status})")
        log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

        save_log({
            **job_log,
            "status": existing.status,
            "resumed_at": datetime.now().isoformat()
        })

        job_id = existing.id


# ── START NEW JOB ──────────────────────────────────────────────────────────────

if not job_id:

    train_n = count_lines(TRAIN_PATH)
    val_n   = count_lines(VAL_PATH)

    log(f"Train rows: {train_n}  |  Val rows: {val_n}")

    log("Uploading files...")

    with open(TRAIN_PATH, "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")

    with open(VAL_PATH, "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")

    log(f"Train file id: {train_file.id}")
    log(f"Val file id:   {val_file.id}")

    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=MODEL,
        hyperparameters={
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate_multiplier": LR_MULT
        },
        suffix=SUFFIX
    )

    job_id = job.id

    log(f"✅ Job launched: {job_id}")
    log(f"OpenAI UI: https://platform.openai.com/finetune/{job_id}")

    save_log({
        "job_id": job_id,
        "model": MODEL,
        "train_rows": train_n,
        "val_rows": val_n,
        "train_file_id": train_file.id,
        "val_file_id": val_file.id,
        "hyperparameters": {
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr_mult": LR_MULT
        },
        "suffix": SUFFIX,
        "launched_at": datetime.now().isoformat(),
        "status": job.status
    })


# ── CHECK INITIAL STATUS ───────────────────────────────────────────────────────

latest = client.fine_tuning.jobs.retrieve(job_id)

log(f"Initial status: {latest.status}")
log("Done. You can run other code now.")

[11:16:19] Previous job ftjob-morNUuf0dpOA07pRxVDfl1f2 already cancelled — launching new job.
[11:16:19] Train rows: 240  |  Val rows: 60
[11:16:19] Uploading files...
[11:16:22] Train file id: file-XhHJ5LqXPxfWoXLjjgLqu3
[11:16:22] Val file id:   file-NAi19MYWcXfuHkdHuN3RhG
[11:16:24] ✅ Job launched: ftjob-w0mN9TSE29SlsnQNXstefj5L
[11:16:24] OpenAI UI: https://platform.openai.com/finetune/ftjob-w0mN9TSE29SlsnQNXstefj5L
[11:16:24] Initial status: validating_files
[11:16:24] Done. You can run other code now.


In [14]:
# ============================================================
# DW-only — Run MANY OpenAI fine-tuned models sequentially
# - Config-driven via MODEL_CFG
# - Resume-safe
# - JSON repair
# - Separate log subfolder per run
# - End-of-run summary
# ============================================================

warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

# =========================
# CONFIG
# =========================
BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
#BASE_DIR = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"

INPUT_FILE  = os.path.join(BASE_DIR, "output", "speech_classification_results_finetuned_dw_multi_models.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "output", "speech_classification_results_finetuned_dw_multi_25_models.xlsx")
OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

TEMP_DIR = os.path.join(BASE_DIR, "temp", "_openai_inference_logs")

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
RUN_LOG_DIR = os.path.join(TEMP_DIR, f"run_{RUN_TS}")
os.makedirs(RUN_LOG_DIR, exist_ok=True)

CONCURRENCY = 10
BATCH_SIZE  = 80

RESUME_IF_OUTPUT_EXISTS = True
ENABLE_REPAIR_CALL      = True

REASONING_REQUIRED = "YES"

DEBUG_TEST_ONLY = False
DEBUG_N_TEST    = 10
TEST_OUTPUT_SUFFIX = "_TEST"

MAX_TOKENS = 800
TEMPERATURE = 0

# =========================
# MODEL CONFIG
# =========================
MODEL_CFG = {
    "gpt41_rag200": {
        "enabled": True,
        "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag200:DKuxbte7",
        "score_col": "DW_gpt41_rag200_score",
        "reason_col": "DW_gpt41_rag200_reason",
    },
    "gpt41_hybrid200": {
        "enabled": True,
        "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag-hybrid200:DL4St05u",
        "score_col": "DW_gpt41_hybrid200_score",
        "reason_col": "DW_gpt41_hybrid200_reason",
    },
    "gpt41_hybrid300": {
        "enabled": True,
        "model": "ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag-hybrid300:DL52CEDZ",
        "score_col": "DW_gpt41_hybrid300_score",
        "reason_col": "DW_gpt41_hybrid300_reason",
    },
}


def safe_test_output_path(path: str) -> str:
    root, ext = os.path.splitext(path)
    return f"{root}{TEST_OUTPUT_SUFFIX}{ext}"

# =========================
# Load API key + client
# =========================
with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()
if not OPENAI_API_KEY:
    raise RuntimeError("openai_api.txt is empty")

client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content. Be attentive to party dynamics in such cases.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{
"score": <integer between 1 and 5 inclusive>, "reason": <string or null>
}

RULES FOR "reason":
- You will be told whether REASONING is REQUIRED via the user message.
- If REASONING IS REQUIRED: set "reason" to a 3-5 sentence explanation that:
  - Includes at least one direct quoted fragment from the passage.
  - Explicitly connects the quoted language to the rubric criteria for DW.
  - Briefly explains why a higher or lower score was not chosen.
- If REASONING IS NOT REQUIRED: "reason" must be exactly null (not the string "null", not "", not "N/A").
""".strip()


USER_TEMPLATE = """
REASONING_REQUIRED: {reasoning_required}
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# RUN STATS
# =========================
RUN_STATS = {name: Counter() for name in MODEL_CFG.keys()}

# =========================
# Logging
# =========================
def save_inference_log(
    model_name: str,
    row_idx: int,
    user_msg: str,
    raw_output: str,
    status: str,  # repaired | failed | api_error
    broken_output: Optional[str] = None,
    note: Optional[str] = None,
) -> None:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    obj = {
        "model": model_name,
        "row_index": row_idx,
        "timestamp_utc": ts,
        "status": status,
        "user_prompt": user_msg,
        "raw_output": raw_output,
    }
    if broken_output is not None:
        obj["broken_output"] = broken_output
    if note is not None:
        obj["note"] = note

    fname = f"{model_name}_row{row_idx:05d}_{status}_{ts}.json"
    with open(os.path.join(RUN_LOG_DIR, fname), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# =========================
# Helpers
# =========================
def _read_tabular(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm", ".xlsb"):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def save_full(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def build_user_message(row: pd.Series) -> str:
    party = (str(row.get("party", "")) or "UNKNOWN").strip()
    passage = str(row.get("paragraph_clean", "")).strip()
    return USER_TEMPLATE.format(
        reasoning_required=REASONING_REQUIRED,
        party=party,
        passage=passage
    )

def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = str(text).strip()

    if t.startswith("```"):
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
        if t.startswith("json"):
            t = t[4:]
        t = t.strip()

    try:
        obj = json.loads(t)
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        pass

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None

    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        return None

def code_repair_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    t = re.sub(r"```json|```", "", str(text)).strip()
    m = re.search(r"\{.*", t, flags=re.DOTALL)
    if not m:
        return None

    t = m.group(0)
    t = re.sub(r",\s*([}\]])", r"\1", t)

    if t.count('"') % 2 != 0:
        t += '"'
    t += "}" * max(0, t.count("{") - t.count("}"))
    t += "]" * max(0, t.count("[") - t.count("]"))

    try:
        return json.loads(t)
    except Exception:
        pass

    score_m  = re.search(r'"score"\s*:\s*([1-5])', t)
    reason_m = re.search(r'"reason"\s*:\s*"(.*?)(?:"|$)', t, flags=re.DOTALL)
    if score_m:
        return {
            "score": int(score_m.group(1)),
            "reason": reason_m.group(1).strip() if reason_m else None,
        }
    return None

def validate_dw_reason(obj: Dict[str, Any]) -> Tuple[bool, str, Optional[int], Optional[str]]:
    if not isinstance(obj, dict):
        return False, "Not a dict", None, None

    if "score" not in obj or "reason" not in obj:
        return False, "Missing score/reason", None, None

    s = obj["score"]
    r = obj["reason"]

    if isinstance(s, str) and s.strip().isdigit():
        s = int(s.strip())

    if not isinstance(s, int) or not (1 <= s <= 5):
        return False, f"Bad score: {s!r}", None, None

    if REASONING_REQUIRED == "YES":
        if not isinstance(r, str) or len(r.strip()) < 10:
            return False, f"Bad reason: {type(r).__name__}", None, None
        return True, "OK", s, r.strip()

    return True, "OK", s, None

async def with_retries(call_fn, meta: str, max_retries=10, base_wait=2, max_wait=120):
    last_err = None
    for i in range(max_retries):
        try:
            return await call_fn()
        except Exception as e:
            last_err = e
            msg = str(e)
            low = msg.lower()

            is_429 = "429" in low or "rate limit" in low
            is_transient = (
                is_429 or
                "503" in low or "overload" in low or "unavailable" in low or
                "timeout" in low or "timed out" in low or "deadline" in low or
                "temporarily" in low or "try again" in low
            )
            if not is_transient:
                return f"Error:{msg[:300]}"

            base = 15 if is_429 else base_wait
            wait = min(max_wait, base * (2 ** i)) + random.uniform(0, 1.5)
            print(f"\n⚠️ Transient error ({meta}): {msg[:120]} | retry in {wait:.1f}s")
            await asyncio.sleep(wait)

    return f"Error:MaxRetries:{meta}:{str(last_err)[:200]}"

# =========================
# Model helpers
# =========================
def enabled_models() -> Dict[str, Dict[str, str]]:
    return {k: v for k, v in MODEL_CFG.items() if v.get("enabled", True)}

def all_output_cols() -> List[str]:
    cols = []
    for cfg in MODEL_CFG.values():
        cols.extend([cfg["score_col"], cfg["reason_col"]])
    return cols

def ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in all_output_cols():
        if c not in df.columns:
            df[c] = ""
    return df

def row_done(df: pd.DataFrame, i: int, model_name: str) -> bool:
    cfg = MODEL_CFG[model_name]
    score_col = cfg["score_col"]
    reason_col = cfg["reason_col"]
    return (
        str(df.at[i, score_col]).strip() != "" and
        str(df.at[i, reason_col]).strip() != ""
    )

def resume_merge(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    if not os.path.exists(out_path):
        return ensure_cols(df)

    old = _read_tabular(out_path)
    old = ensure_cols(old)
    df  = ensure_cols(df)

    key = None
    for k in ["q_id", "ParagraphID"]:
        if k in df.columns and k in old.columns:
            key = k
            break
    if key is None:
        return df

    df[key]  = df[key].astype(str).str.strip()
    old[key] = old[key].astype(str).str.strip()

    old = old.sort_values(key).drop_duplicates(key, keep="last")

    keep_cols = [key] + all_output_cols()
    old = old[[c for c in keep_cols if c in old.columns]]

    df = df.merge(old, on=key, how="left", suffixes=("", "_old"))

    for c in all_output_cols():
        oldc = f"{c}_old"
        if oldc in df.columns:
            df[c] = df[c].where(df[c].astype(str).str.strip() != "", df[oldc].fillna(""))
            df.drop(columns=[oldc], inplace=True)

    return df

# =========================
# Inference
# =========================
async def call_model(model_id: str, user_msg: str) -> str:
    # GPT-5 family -> Responses API
    if model_id.startswith("gpt-5"):
        resp = await client.responses.create(
            model=model_id,
            input=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            reasoning={"effort": "none"},   # or "low"
            text={"verbosity": "low"},
            max_output_tokens=4000,         # start here; increase if needed
        )
        return (resp.output_text or "").strip()

    # GPT-4.1 family -> Chat Completions API
    else:
        resp = await client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            temperature=TEMPERATURE,
            max_tokens=800,
        )
        return (resp.choices[0].message.content or "").strip()
    

async def score_dw_with_reason(
    model_name: str,
    row: pd.Series,
    sema: asyncio.Semaphore,
    meta: str,
    row_idx: int,
) -> Dict[str, Any]:
    async with sema:
        user_msg = build_user_message(row)
        model_id = MODEL_CFG[model_name]["model"]

        async def _primary():
            return await call_model(model_id, user_msg)

        res = await with_retries(_primary, meta=meta)
        if isinstance(res, str) and res.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "api_error")
            return {"score": None, "reason": res}

        obj = extract_json_from_text(res) or code_repair_json(res)
        if obj is not None:
            ok, _, s, r = validate_dw_reason(obj)
            if ok:
                RUN_STATS[model_name]["ok"] += 1
                return {"score": s, "reason": r}

        if not ENABLE_REPAIR_CALL:
            RUN_STATS[model_name]["failed"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "failed", note="No repair call enabled.")
            return {"score": None, "reason": "INVALID_JSON_NO_REPAIR"}

        obj_pre = extract_json_from_text(res) or code_repair_json(res)
        fail_reason = validate_dw_reason(obj_pre)[1] if obj_pre is not None else "unparseable JSON"

        repair_system = (
            "Fix the following output into VALID JSON ONLY.\n"
            'Required format: {"score": 3, "reason": "3-5 sentences with a direct quote fragment."}\n'
            "No markdown. No extra text. No extra keys."
        )
        repair_user = (
            f"Your previous output failed validation. Reason: {fail_reason}\n\n"
            f"Broken output:\n{res}"
        )

        async def _repair():
            resp = await client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": repair_system},
                    {"role": "user",   "content": repair_user},
                ],
                temperature=0,
                max_tokens=MAX_TOKENS,
            )
            return (resp.choices[0].message.content or "").strip()

        res2 = await with_retries(_repair, meta=meta + "|repair")
        if isinstance(res2, str) and res2.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res2, "api_error", broken_output=res)
            return {"score": None, "reason": res2}

        obj2 = extract_json_from_text(res2) or code_repair_json(res2)
        if obj2 is not None:
            ok2, _, s2, r2 = validate_dw_reason(obj2)
            if ok2:
                RUN_STATS[model_name]["repaired"] += 1
                save_inference_log(
                    model_name, row_idx, user_msg, res2, "repaired",
                    broken_output=res,
                    note="Primary response failed validation and was repaired."
                )
                return {"score": s2, "reason": r2}

        RUN_STATS[model_name]["failed"] += 1
        save_inference_log(
            model_name, row_idx, user_msg, res, "failed",
            note="Failed after repair attempt."
        )
        return {"score": None, "reason": "INVALID_JSON_AFTER_REPAIR"}

# =========================
# Sequential runner
# =========================
async def run_one_model(
    df: pd.DataFrame,
    model_name: str,
    sema: asyncio.Semaphore,
    process_indices: List[int],
) -> None:
    cfg = MODEL_CFG[model_name]
    score_col  = cfg["score_col"]
    reason_col = cfg["reason_col"]

    for k in range(0, len(process_indices), BATCH_SIZE):
        batch = process_indices[k:k + BATCH_SIZE]
        todo  = [i for i in batch if not row_done(df, i, model_name)]
        if not todo:
            continue

        async def run_row(ii: int):
            meta = f"{model_name}|row={ii}"
            obj = await score_dw_with_reason(model_name, df.iloc[ii], sema, meta, ii)
            return ii, obj

        tasks = [asyncio.create_task(run_row(i)) for i in todo]

        batch_num = k // BATCH_SIZE + 1
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"{model_name} batch {batch_num}"):
            i, obj = await fut
            score = obj.get("score")
            reason = obj.get("reason")

            df.at[i, score_col]  = "" if score is None else str(score)
            df.at[i, reason_col] = "" if reason is None else str(reason)

        save_full(df, OUTPUT_PATH)
        print(f"[Checkpoint] Saved → {OUTPUT_PATH}")

# =========================
# Summary
# =========================
def print_summary():
    print("\n================ SUMMARY ================")
    for model_name, cfg in enabled_models().items():
        stats = RUN_STATS[model_name]
        ok = stats.get("ok", 0)
        repaired = stats.get("repaired", 0)
        failed = stats.get("failed", 0)
        api_error = stats.get("api_error", 0)
        total_processed = ok + repaired + failed + api_error

        print(f"{model_name}:")
        print(f"  model       = {cfg['model']}")
        print(f"  processed   = {total_processed}")
        print(f"  ok          = {ok}")
        print(f"  repaired    = {repaired}")
        print(f"  failed      = {failed}")
        print(f"  api_error   = {api_error}")
        print(f"  usable      = {ok + repaired}")
        print()

    print(f"Log folder: {RUN_LOG_DIR}")
    print("=========================================\n")

# =========================
# MAIN
# =========================
async def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(INPUT_FILE)

    global OUTPUT_PATH
    OUTPUT_PATH = safe_test_output_path(OUTPUT_FILE) if DEBUG_TEST_ONLY else OUTPUT_FILE

    df = _read_tabular(INPUT_FILE)
    df = ensure_cols(df)

    if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_PATH):
        df = resume_merge(df, OUTPUT_PATH)

    for c in ["paragraph_clean", "party"]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    active_models = enabled_models()
    if not active_models:
        raise ValueError("No models enabled in MODEL_CFG.")

    n = len(df)
    process_indices = list(range(min(DEBUG_N_TEST, n))) if DEBUG_TEST_ONLY else list(range(n))
    sema = asyncio.Semaphore(CONCURRENCY)

    print(f"Rows: {n:,} | Processing: {len(process_indices):,}")
    print(f"Output: {OUTPUT_PATH}")
    print(f"Logs root: {TEMP_DIR}")
    print(f"Run logs:  {RUN_LOG_DIR}")
    print(f"Test mode: {DEBUG_TEST_ONLY}")
    print(f"Enabled models: {list(active_models.keys())}")

    for model_name, cfg in active_models.items():
        print(f"\n=== Running {model_name} sequentially ===")
        print(cfg["model"])
        await run_one_model(df, model_name, sema, process_indices)

    save_full(df, OUTPUT_PATH)
    print_summary()
    print("✅ DONE")

# RUN
await main()

Rows: 551 | Processing: 551
Output: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx
Logs root: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\_openai_inference_logs
Run logs:  C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\_openai_inference_logs\run_20260319T110401
Test mode: False
Enabled models: ['gpt41_rag200', 'gpt41_hybrid200', 'gpt41_hybrid300']

=== Running gpt41_rag200 sequentially ===
ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag200:DKuxbte7


gpt41_rag200 batch 1: 100%|██████████| 80/80 [00:27<00:00,  2.95it/s]


[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx


gpt41_rag200 batch 2: 100%|██████████| 80/80 [01:08<00:00,  1.17it/s]


[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx


...

[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx


gpt41_hybrid300 batch 6: 100%|██████████| 80/80 [01:05<00:00,  1.21it/s]


[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx


gpt41_hybrid300 batch 7: 100%|██████████| 71/71 [00:47<00:00,  1.49it/s]


[Checkpoint] Saved → C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\output\speech_classification_results_finetuned_dw_multi_25_models.xlsx

================ SUMMARY ================
gpt41_rag200:
  model       = ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag200:DKuxbte7
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

gpt41_hybrid200:
  model       = ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag-hybrid200:DL4St05u
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

gpt41_hybrid300:
  model       = ft:gpt-4.1-2025-04-14:stockholm-university:dw-collusion-ordinal-rag-hybrid300:DL52CEDZ
  processed   = 551
  ok          = 551
  repaired    = 0
  failed      = 0
  api_error   = 0
  usable      = 551

Log folder: C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models\temp\_openai_inf

### mini models 

#### with reasoning 

In [ ]:
# ============================================================
# DW-only — Run MANY OpenAI models sequentially
# - Config-driven via MODEL_CFG
# - Resume-safe
# - JSON schema enforcement (no repair needed)
# - Responses API unified for all models
# - Separate log subfolder per run
# - End-of-run summary
# ============================================================

warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

# =========================
# CONFIG
# =========================
#BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
BASE_DIR = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
INPUT_FILE  = os.path.join(BASE_DIR, "input", "speech_classification_cleaned_dataset.xlsx")
OUTPUT_FILE = os.path.join(BASE_DIR, "output", "speech_classification_results_base_multi_models.xlsx")
OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

TEMP_DIR    = os.path.join(BASE_DIR, "temp", "_openai_inference_logs")
RUN_TS      = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
RUN_LOG_DIR = os.path.join(TEMP_DIR, f"run_{RUN_TS}")
os.makedirs(RUN_LOG_DIR, exist_ok=True)

CONCURRENCY = 10
BATCH_SIZE  = 80

RESUME_IF_OUTPUT_EXISTS = True

# JSON schema enforcement means repair is rarely needed,
# but keep it as a last-resort fallback
ENABLE_REPAIR_CALL = True

REASONING_REQUIRED = "YES"

DEBUG_TEST_ONLY    = False
DEBUG_N_TEST       = 10
TEST_OUTPUT_SUFFIX = "_TEST"

MAX_TOKENS  = 300   # score + 3-5 sentence reason fits comfortably
TEMPERATURE = 0     # used only for GPT-4.1 family (ignored for GPT-5.x)

# =========================
# MODEL CONFIG
# =========================
# "is_reasoning_model": True  → passes reasoning={"effort": "low"}, NO temperature
# "is_reasoning_model": False → passes temperature=TEMPERATURE,    NO reasoning
MODEL_CFG = {
    # ── GPT-5.4 family (reasoning models) ───────────────────────────────
    "gpt54": {
        "enabled": True,
        "model": "gpt-5.4-2026-03-05",
        "is_reasoning_model": True,
        "price_in":  2.50,   # USD per 1M input tokens
        "price_out": 15.00,  # USD per 1M output tokens
        "score_col": "DW_gpt54_score_w_reasoning",
        "reason_col": "DW_gpt54_reason_w_reasoning",
    },
    "gpt54_mini": {
        "enabled": True,
        "model": "gpt-5.4-mini-2026-03-17",
        "is_reasoning_model": True,
        "price_in":  0.75,
        "price_out": 4.5,
        "score_col": "DW_gpt54_mini_score_w_reasoning",
        "reason_col": "DW_gpt54_mini_reason_w_reasoning",
    },
    "gpt54_nano": {
        "enabled": True,
        "model": "gpt-5.4-nano-2026-03-17",
        "is_reasoning_model": True,
        "price_in":  0.20,
        "price_out": 1.25,
        "score_col": "DW_gpt54_nano_score_w_reasoning",
        "reason_col": "DW_gpt54_nano_reason_w_reasoning",
    },
    # ── GPT-4.1 family (non-reasoning models) ───────────────────────────
    "gpt41": {
        "enabled": True,
        "model": "gpt-4.1-2025-04-14",
        "is_reasoning_model": False,
        "price_in":  2.00,
        "price_out": 8.00,
        "score_col": "DW_gpt41_score_w_reasoning",
        "reason_col": "DW_gpt41_reason_w_reasoning",
    },
    "gpt41_mini": {
        "enabled": True,
        "model": "gpt-4.1-mini-2025-04-14",
        "is_reasoning_model": False,
        "price_in":  0.40,
        "price_out": 1.60,
        "score_col": "DW_gpt41_mini_score_w_reasoning",
        "reason_col": "DW_gpt41_mini_reason_w_reasoning",
    },
    "gpt41_nano": {
        "enabled": True,
        "model": "gpt-4.1-nano-2025-04-14",
        "is_reasoning_model": False,
        "price_in":  0.10,
        "price_out": 0.40,
        "score_col": "DW_gpt41_nano_score_w_reasoning",
        "reason_col": "DW_gpt41_nano_reason_w_reasoning",
    },
}

# JSON schema enforced on every call — eliminates most repair needs
DW_JSON_SCHEMA = {
    "format": {
        "type": "json_schema",
        "name": "dw_score",       # name goes directly here
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "score":  {"type": "integer"},
                "reason": {"type": ["string", "null"]},
            },
            "required": ["score", "reason"],
            "additionalProperties": False,
        },
    }
}

def safe_test_output_path(path: str) -> str:
    root, ext = os.path.splitext(path)
    return f"{root}{TEST_OUTPUT_SUFFIX}{ext}"

# =========================
# Load API key + client
# =========================
with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
    OPENAI_API_KEY = f.read().strip()
if not OPENAI_API_KEY:
    raise RuntimeError("openai_api.txt is empty")

client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content. Be attentive to party dynamics in such cases.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{
"score": <integer between 1 and 5 inclusive>, "reason": <string or null>
}

RULES FOR "reason":
- You will be told whether REASONING is REQUIRED via the user message.
- If REASONING IS REQUIRED: set "reason" to a 3-5 sentence explanation that:
  - Includes at least one direct quoted fragment from the passage.
  - Explicitly connects the quoted language to the rubric criteria for DW.
  - Briefly explains why a higher or lower score was not chosen.
- If REASONING IS NOT REQUIRED: "reason" must be exactly null (not the string "null", not "", not "N/A").
""".strip()

USER_TEMPLATE = """
REASONING_REQUIRED: {reasoning_required}
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# RUN STATS
# =========================
RUN_STATS = {name: Counter() for name in MODEL_CFG.keys()}

# =========================
# Logging
# =========================
def save_inference_log(
    model_name: str,
    row_idx: int,
    user_msg: str,
    raw_output: str,
    status: str,
    broken_output: Optional[str] = None,
    note: Optional[str] = None,
) -> None:
    ts  = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    obj = {
        "model":         model_name,
        "row_index":     row_idx,
        "timestamp_utc": ts,
        "status":        status,
        "user_prompt":   user_msg,
        "raw_output":    raw_output,
    }
    if broken_output is not None:
        obj["broken_output"] = broken_output
    if note is not None:
        obj["note"] = note

    fname = f"{model_name}_row{row_idx:05d}_{status}_{ts}.json"
    with open(os.path.join(RUN_LOG_DIR, fname), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# =========================
# Helpers
# =========================
def _read_tabular(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm", ".xlsb"):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def save_full(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def build_user_message(row: pd.Series) -> str:
    party   = (str(row.get("party", "")) or "UNKNOWN").strip()
    passage = str(row.get("paragraph_clean", "")).strip()
    return USER_TEMPLATE.format(
        reasoning_required=REASONING_REQUIRED,
        party=party,
        passage=passage,
    )

def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    """Parse JSON from model output — first line of defence."""
    if not text:
        return None
    t = str(text).strip()

    if t.startswith("```"):
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
        if t.startswith("json"):
            t = t[4:]
        t = t.strip()

    try:
        obj = json.loads(t)
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        pass

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        return None

def code_repair_json(text: str) -> Optional[Dict[str, Any]]:
    """Last-resort structural repair before calling the repair API."""
    if not text:
        return None
    t = re.sub(r"```json|```", "", str(text)).strip()
    m = re.search(r"\{.*", t, flags=re.DOTALL)
    if not m:
        return None
    t = m.group(0)
    t = re.sub(r",\s*([}\]])", r"\1", t)
    if t.count('"') % 2 != 0:
        t += '"'
    t += "}" * max(0, t.count("{") - t.count("}"))
    t += "]" * max(0, t.count("[") - t.count("]"))
    try:
        return json.loads(t)
    except Exception:
        pass
    score_m  = re.search(r'"score"\s*:\s*([1-5])', t)
    reason_m = re.search(r'"reason"\s*:\s*"(.*?)(?:"|$)', t, flags=re.DOTALL)
    if score_m:
        return {
            "score":  int(score_m.group(1)),
            "reason": reason_m.group(1).strip() if reason_m else None,
        }
    return None

def validate_dw_reason(obj: Dict[str, Any]) -> Tuple[bool, str, Optional[int], Optional[str]]:
    if not isinstance(obj, dict):
        return False, "Not a dict", None, None
    if "score" not in obj or "reason" not in obj:
        return False, "Missing score/reason", None, None

    s = obj["score"]
    r = obj["reason"]

    if isinstance(s, str) and s.strip().isdigit():
        s = int(s.strip())
    if not isinstance(s, int) or not (1 <= s <= 5):
        return False, f"Bad score: {s!r}", None, None

    if REASONING_REQUIRED == "YES":
        if not isinstance(r, str) or len(r.strip()) < 10:
            return False, f"Bad reason: {type(r).__name__}", None, None
        return True, "OK", s, r.strip()

    return True, "OK", s, None

async def with_retries(call_fn, meta: str, max_retries=10, base_wait=2, max_wait=120):
    last_err = None
    for i in range(max_retries):
        try:
            return await call_fn()
        except Exception as e:
            last_err = e
            msg = str(e)
            low = msg.lower()
            is_429      = "429" in low or "rate limit" in low
            is_transient = (
                is_429 or
                "503" in low or "overload" in low or "unavailable" in low or
                "timeout" in low or "timed out" in low or "deadline" in low or
                "temporarily" in low or "try again" in low
            )
            if not is_transient:
                return f"Error:{msg[:300]}"
            base = 15 if is_429 else base_wait
            wait = min(max_wait, base * (2 ** i)) + random.uniform(0, 1.5)
            print(f"\n⚠️  Transient error ({meta}): {msg[:120]} | retry in {wait:.1f}s")
            await asyncio.sleep(wait)
    return f"Error:MaxRetries:{meta}:{str(last_err)[:200]}"

# =========================
# Model helpers
# =========================
def enabled_models() -> Dict[str, Dict]:
    return {k: v for k, v in MODEL_CFG.items() if v.get("enabled", True)}

def all_output_cols() -> List[str]:
    cols = []
    for cfg in MODEL_CFG.values():
        cols.extend([cfg["score_col"], cfg["reason_col"]])
    return cols

def ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in all_output_cols():
        if c not in df.columns:
            df[c] = ""
    return df

def row_done(df: pd.DataFrame, i: int, model_name: str) -> bool:
    cfg = MODEL_CFG[model_name]
    return (
        str(df.at[i, cfg["score_col"]]).strip() != "" and
        str(df.at[i, cfg["reason_col"]]).strip() != ""
    )

def resume_merge(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    if not os.path.exists(out_path):
        return ensure_cols(df)
    old = _read_tabular(out_path)
    old = ensure_cols(old)
    df  = ensure_cols(df)
    key = None
    for k in ["q_id", "ParagraphID"]:
        if k in df.columns and k in old.columns:
            key = k
            break
    if key is None:
        return df
    df[key]  = df[key].astype(str).str.strip()
    old[key] = old[key].astype(str).str.strip()
    old = old.sort_values(key).drop_duplicates(key, keep="last")
    keep_cols = [key] + all_output_cols()
    old = old[[c for c in keep_cols if c in old.columns]]
    df  = df.merge(old, on=key, how="left", suffixes=("", "_old"))
    for c in all_output_cols():
        oldc = f"{c}_old"
        if oldc in df.columns:
            df[c] = df[c].where(df[c].astype(str).str.strip() != "", df[oldc].fillna(""))
            df.drop(columns=[oldc], inplace=True)
    return df

# =========================
# Unified Responses API call
# =========================
async def call_model(
    model_id: str, user_msg: str, is_reasoning_model: bool
) -> Tuple[str, int, int]:
    """
    Returns: (output_text, input_tokens, output_tokens)
    """
    kwargs = dict(
        model=model_id,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        text=DW_JSON_SCHEMA,
        max_output_tokens=MAX_TOKENS,
    )
    if is_reasoning_model:
        kwargs["reasoning"] = {"effort": "low"}
    else:
        kwargs["temperature"] = TEMPERATURE

    resp = await client.responses.create(**kwargs)
    text        = (resp.output_text or "").strip()
    in_tokens   = resp.usage.input_tokens  if resp.usage else 0
    out_tokens  = resp.usage.output_tokens if resp.usage else 0
    return text, in_tokens, out_tokens


async def call_model_repair(
    model_id: str, repair_system: str, repair_user: str, is_reasoning_model: bool
) -> Tuple[str, int, int]:
    """
    Returns: (output_text, input_tokens, output_tokens)
    """
    kwargs = dict(
        model=model_id,
        input=[
            {"role": "system", "content": repair_system},
            {"role": "user",   "content": repair_user},
        ],
        max_output_tokens=MAX_TOKENS,
    )
    if is_reasoning_model:
        kwargs["reasoning"] = {"effort": "none"}
    else:
        kwargs["temperature"] = 0

    resp = await client.responses.create(**kwargs)
    text       = (resp.output_text or "").strip()
    in_tokens  = resp.usage.input_tokens  if resp.usage else 0
    out_tokens = resp.usage.output_tokens if resp.usage else 0
    return text, in_tokens, out_tokens

# =========================
# Inference
# =========================
async def score_dw_with_reason(
    model_name: str,
    row: pd.Series,
    sema: asyncio.Semaphore,
    meta: str,
    row_idx: int,
) -> Dict[str, Any]:
    async with sema:
        user_msg           = build_user_message(row)
        cfg                = MODEL_CFG[model_name]
        model_id           = cfg["model"]
        is_reasoning_model = cfg["is_reasoning_model"]

        # ── Primary call ──────────────────────────────────────────────
        async def _primary():
            return await call_model(model_id, user_msg, is_reasoning_model)

        res_tuple = await with_retries(_primary, meta=meta)

        # with_retries returns a plain error string on failure
        if isinstance(res_tuple, str) and res_tuple.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res_tuple, "api_error")
            return {"score": None, "reason": res_tuple}

        res, in_tok, out_tok = res_tuple
        RUN_STATS[model_name]["input_tokens"]  += in_tok
        RUN_STATS[model_name]["output_tokens"] += out_tok

        obj = extract_json_from_text(res) or code_repair_json(res)
        if obj is not None:
            ok, _, s, r = validate_dw_reason(obj)
            if ok:
                RUN_STATS[model_name]["ok"] += 1
                return {"score": s, "reason": r}

        # ── Repair call ───────────────────────────────────────────────
        if not ENABLE_REPAIR_CALL:
            RUN_STATS[model_name]["failed"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "failed",
                               note="No repair call enabled.")
            return {"score": None, "reason": "INVALID_JSON_NO_REPAIR"}

        obj_pre     = extract_json_from_text(res) or code_repair_json(res)
        fail_reason = validate_dw_reason(obj_pre)[1] if obj_pre is not None else "unparseable JSON"

        repair_system = (
            "Fix the following output into VALID JSON ONLY.\n"
            'Required format: {"score": 3, "reason": "3-5 sentences with a direct quote fragment."}\n'
            "No markdown. No extra text. No extra keys."
        )
        repair_user_msg = (
            f"Your previous output failed validation. Reason: {fail_reason}\n\n"
            f"Broken output:\n{res}"
        )

        async def _repair():
            return await call_model_repair(
                model_id, repair_system, repair_user_msg, is_reasoning_model
            )

        res2_tuple = await with_retries(_repair, meta=meta + "|repair")

        if isinstance(res2_tuple, str) and res2_tuple.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res2_tuple, "api_error",
                               broken_output=res)
            return {"score": None, "reason": res2_tuple}

        res2, in_tok2, out_tok2 = res2_tuple
        RUN_STATS[model_name]["input_tokens"]  += in_tok2
        RUN_STATS[model_name]["output_tokens"] += out_tok2

        obj2 = extract_json_from_text(res2) or code_repair_json(res2)
        if obj2 is not None:
            ok2, _, s2, r2 = validate_dw_reason(obj2)
            if ok2:
                RUN_STATS[model_name]["repaired"] += 1
                save_inference_log(model_name, row_idx, user_msg, res2, "repaired",
                                   broken_output=res,
                                   note="Primary response failed; repaired successfully.")
                return {"score": s2, "reason": r2}

        RUN_STATS[model_name]["failed"] += 1
        save_inference_log(model_name, row_idx, user_msg, res, "failed",
                           note="Failed after repair attempt.")
        return {"score": None, "reason": "INVALID_JSON_AFTER_REPAIR"}

# =========================
# Sequential runner
# =========================
async def run_one_model(
    df: pd.DataFrame,
    model_name: str,
    sema: asyncio.Semaphore,
    process_indices: List[int],
) -> None:
    cfg        = MODEL_CFG[model_name]
    score_col  = cfg["score_col"]
    reason_col = cfg["reason_col"]

    for k in range(0, len(process_indices), BATCH_SIZE):
        batch = process_indices[k:k + BATCH_SIZE]
        todo  = [i for i in batch if not row_done(df, i, model_name)]
        if not todo:
            continue

        async def run_row(ii: int):
            meta = f"{model_name}|row={ii}"
            obj  = await score_dw_with_reason(model_name, df.iloc[ii], sema, meta, ii)
            return ii, obj

        tasks = [asyncio.create_task(run_row(i)) for i in todo]

        batch_num = k // BATCH_SIZE + 1
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks),
                        desc=f"{model_name} batch {batch_num}"):
            i, obj = await fut
            score  = obj.get("score")
            reason = obj.get("reason")
            df.at[i, score_col]  = "" if score  is None else str(score)
            df.at[i, reason_col] = "" if reason is None else str(reason)

        save_full(df, OUTPUT_PATH)
        print(f"[Checkpoint] Saved → {OUTPUT_PATH}")

    # ── Per-model cost summary ────────────────────────────────────────
    stats                   = RUN_STATS[model_name]
    in_tok                  = stats.get("input_tokens",  0)
    out_tok                 = stats.get("output_tokens", 0)
    in_cost, out_cost, total = compute_cost(model_name)

    print(f"\n{'='*55}")
    print(f"  ✅ FINISHED: {model_name}  [{cfg['model']}]")
    print(f"{'='*55}")
    print(f"  Tokens   — input:  {in_tok:>12,}")
    print(f"             output: {out_tok:>12,}")
    print(f"  Prices   — input:  ${cfg['price_in']:.2f}/1M   output: ${cfg['price_out']:.2f}/1M")
    print(f"  Cost     — input:  ${in_cost:>10.4f}")
    print(f"             output: ${out_cost:>10.4f}")
    print(f"             TOTAL:  ${total:>10.4f}")
    print(f"{'='*55}\n")

# =========================
# Summary
# =========================
def compute_cost(model_name: str) -> Tuple[float, float, float]:
    """Returns (input_cost_usd, output_cost_usd, total_cost_usd)."""
    cfg       = MODEL_CFG[model_name]
    stats     = RUN_STATS[model_name]
    in_tok    = stats.get("input_tokens",  0)
    out_tok   = stats.get("output_tokens", 0)
    in_cost   = in_tok  / 1_000_000 * cfg["price_in"]
    out_cost  = out_tok / 1_000_000 * cfg["price_out"]
    return in_cost, out_cost, in_cost + out_cost

def save_summary_log() -> None:
    ts      = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    fname   = f"summary_{ts}.txt"
    fpath   = os.path.join(RUN_LOG_DIR, fname)

    lines = []
    lines.append("=" * 65)
    lines.append("  FINAL SUMMARY — ALL MODELS")
    lines.append("=" * 65)

    grand_in_cost = grand_out_cost = grand_total = 0.0

    for model_name, cfg in enabled_models().items():
        stats      = RUN_STATS[model_name]
        ok         = stats.get("ok",            0)
        repaired   = stats.get("repaired",      0)
        failed     = stats.get("failed",        0)
        api_error  = stats.get("api_error",     0)
        in_tok     = stats.get("input_tokens",  0)
        out_tok    = stats.get("output_tokens", 0)
        total_rows = ok + repaired + failed + api_error
        in_cost, out_cost, total_cost = compute_cost(model_name)

        grand_in_cost  += in_cost
        grand_out_cost += out_cost
        grand_total    += total_cost

        lines.append(f"\n  {model_name}  [{cfg['model']}]")
        lines.append(f"    rows processed : {total_rows:,}  (ok={ok}, repaired={repaired}, failed={failed}, api_error={api_error})")
        lines.append(f"    input  tokens  : {in_tok:,}  → ${in_cost:.4f}")
        lines.append(f"    output tokens  : {out_tok:,}  → ${out_cost:.4f}")
        lines.append(f"    model cost     : ${total_cost:.4f}")

    lines.append(f"\n{'─' * 65}")
    lines.append("  GRAND TOTAL")
    lines.append(f"    input cost  : ${grand_in_cost:.4f}")
    lines.append(f"    output cost : ${grand_out_cost:.4f}")
    lines.append(f"    TOTAL COST  : ${grand_total:.4f}")
    lines.append(f"\n  Run timestamp : {ts}")
    lines.append(f"  Log folder    : {RUN_LOG_DIR}")
    lines.append(f"  Output file   : {OUTPUT_PATH}")
    lines.append("=" * 65)

    with open(fpath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Summary saved → {fpath}")

def print_summary():
    print("\n" + "="*65)
    print("  FINAL SUMMARY — ALL MODELS")
    print("="*65)

    grand_in_cost = grand_out_cost = grand_total = 0.0

    for model_name, cfg in enabled_models().items():
        stats         = RUN_STATS[model_name]
        ok            = stats.get("ok",            0)
        repaired      = stats.get("repaired",      0)
        failed        = stats.get("failed",        0)
        api_error     = stats.get("api_error",     0)
        in_tok        = stats.get("input_tokens",  0)
        out_tok       = stats.get("output_tokens", 0)
        total_rows    = ok + repaired + failed + api_error
        in_cost, out_cost, total_cost = compute_cost(model_name)

        grand_in_cost  += in_cost
        grand_out_cost += out_cost
        grand_total    += total_cost

        print(f"\n  {model_name}  [{cfg['model']}]")
        print(f"    rows processed : {total_rows:,}  (ok={ok}, repaired={repaired}, failed={failed}, api_error={api_error})")
        print(f"    input  tokens  : {in_tok:,}  → ${in_cost:.4f}")
        print(f"    output tokens  : {out_tok:,}  → ${out_cost:.4f}")
        print(f"    model cost     : ${total_cost:.4f}")

    print(f"\n{'─'*65}")
    print(f"  GRAND TOTAL")
    print(f"    input cost  : ${grand_in_cost:.4f}")
    print(f"    output cost : ${grand_out_cost:.4f}")
    print(f"    TOTAL COST  : ${grand_total:.4f}")
    print(f"\n  Log folder: {RUN_LOG_DIR}")
    print("="*65 + "\n")
    save_summary_log()

# =========================
# MAIN
# =========================
async def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(INPUT_FILE)

    global OUTPUT_PATH
    OUTPUT_PATH = safe_test_output_path(OUTPUT_FILE) if DEBUG_TEST_ONLY else OUTPUT_FILE

    df = _read_tabular(INPUT_FILE)
    df = ensure_cols(df)

    if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_PATH):
        df = resume_merge(df, OUTPUT_PATH)

    for c in ["paragraph_clean", "party"]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    active_models = enabled_models()
    if not active_models:
        raise ValueError("No models enabled in MODEL_CFG.")

    n               = len(df)
    process_indices = list(range(min(DEBUG_N_TEST, n))) if DEBUG_TEST_ONLY else list(range(n))
    sema            = asyncio.Semaphore(CONCURRENCY)

    print(f"Rows: {n:,} | Processing: {len(process_indices):,}")
    print(f"Output:     {OUTPUT_PATH}")
    print(f"Run logs:   {RUN_LOG_DIR}")
    print(f"Test mode:  {DEBUG_TEST_ONLY}")
    print(f"Enabled models: {list(active_models.keys())}")

    for model_name, cfg in active_models.items():
        print(f"\n=== Running {model_name}  [{cfg['model']}] ===")
        await run_one_model(df, model_name, sema, process_indices)

    save_full(df, OUTPUT_PATH)
    print_summary()
    print("✅ DONE")

# RUN
await main()

Rows: 511 | Processing: 511
Output:     /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx
Run logs:   /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/temp/_openai_inference_logs/run_20260405T153345
Test mode:  False
Enabled models: ['gpt54', 'gpt54_mini', 'gpt54_nano', 'gpt41', 'gpt41_mini', 'gpt41_nano']

=== Running gpt54  [gpt-5.4-2026-03-05] ===


gpt54 batch 1: 100%|██████████| 80/80 [00:39<00:00,  2.04it/s]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx


gpt54 batch 2: 100%|██████████| 80/80 [00:36<00:00,  2.19it/s]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx


...

[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx


gpt41_nano batch 6: 100%|██████████| 80/80 [00:10<00:00,  7.42it/s]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx


gpt41_nano batch 7: 100%|██████████| 31/31 [00:04<00:00,  6.66it/s]


[Checkpoint] Saved → /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models.xlsx

  ✅ FINISHED: gpt41_nano  [gpt-4.1-nano-2025-04-14]
  Tokens   — input:       358,236
             output:       50,063
  Prices   — input:  $0.10/1M   output: $0.40/1M
  Cost     — input:  $    0.0358
             output: $    0.0200
             TOTAL:  $    0.0558


  FINAL SUMMARY — ALL MODELS

  gpt54  [gpt-5.4-2026-03-05]
    rows processed : 511  (ok=511, repaired=0, failed=0, api_error=0)
    input  tokens  : 356,783  → $0.8920
    output tokens  : 104,906  → $1.5736
    model cost     : $2.4655

  gpt54_mini  [gpt-5.4-mini-2026-03-17]
    rows processed : 511  (ok=511, repaired=0, failed=0, api_error=0)
    input  tokens  : 356,783  → $0.2676
    output tokens  : 90,908  → $0.4091
    model cost     : $0.6767

  gpt54_nano  [gpt-5.4-nano-2026-03-17]
    rows processed : 511  (ok=511, repaired=0, failed=0, api_e

#### without reasoning

In [9]:
# ============================================================
# DW-only — Run MANY OpenAI models sequentially
# - Config-driven via MODEL_CFG
# - Resume-safe
# - JSON schema enforcement (no repair needed)
# - Responses API unified for all models
# - Separate log subfolder per run
# - End-of-run summary
# ============================================================

import os
import re
import json
import random
import asyncio
import warnings
from datetime import datetime, timezone
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import nest_asyncio
from tqdm import tqdm
from openai import AsyncOpenAI

warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
nest_asyncio.apply()

# =========================
# CONFIG
# =========================
# BASE_DIR = r"C:\Users\alir5807\Dropbox\Conspiratorial_speech\Ali\fine_tuning_models"
BASE_DIR        = "/Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models"
INPUT_FILE      = os.path.join(BASE_DIR, "output", "speech_classification_results_base_multi_models.xlsx")
OUTPUT_FILE     = os.path.join(BASE_DIR, "output", "speech_classification_results_base_multi_models_2.xlsx")
OPENAI_KEY_PATH = os.path.join(BASE_DIR, "input", "api's", "openai_api.txt")

TEMP_DIR    = os.path.join(BASE_DIR, "temp", "_openai_inference_logs")
RUN_TS      = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
RUN_LOG_DIR = os.path.join(TEMP_DIR, f"run_{RUN_TS}")
os.makedirs(RUN_LOG_DIR, exist_ok=True)

CONCURRENCY = 10
BATCH_SIZE  = 80

RESUME_IF_OUTPUT_EXISTS = True
ENABLE_REPAIR_CALL      = True

DEBUG_TEST_ONLY    = False
DEBUG_N_TEST       = 10
TEST_OUTPUT_SUFFIX = "_TEST"

TEMPERATURE = 0  # only used for non-reasoning models (gpt-4.1 family)

# =========================
# MODEL CONFIG
# =========================
# is_reasoning_model: True  → passes reasoning={"effort": "low"}, NO temperature
# is_reasoning_model: False → passes temperature=TEMPERATURE,     NO reasoning
MODEL_CFG = {
    "gpt54": {
        "enabled":            True,
        "model":              "gpt-5.4-2026-03-05",
        "is_reasoning_model": True,
        "price_in":           2.50,
        "price_out":          15.00,
        "score_col":          "DW_gpt54_score",
        "max_tokens":         300,
    },
    "gpt54_mini": {
        "enabled":            True,
        "model":              "gpt-5.4-mini-2026-03-17",
        "is_reasoning_model": True,
        "price_in":           0.75,
        "price_out":          4.50,
        "score_col":          "DW_gpt54_mini_score",
        "max_tokens":         300,
    },
    "gpt54_nano": {
        "enabled":            True,
        "model":              "gpt-5.4-nano-2026-03-17",
        "is_reasoning_model": True,
        "price_in":           0.20,
        "price_out":          1.25,
        "score_col":          "DW_gpt54_nano_score",
        "max_tokens":         300,
    },
    "gpt41": {
        "enabled":            True,
        "model":              "gpt-4.1-2025-04-14",
        "is_reasoning_model": False,
        "price_in":           2.00,
        "price_out":          8.00,
        "score_col":          "DW_gpt41_score",
        "max_tokens":         30,
    },
    "gpt41_mini": {
        "enabled":            True,
        "model":              "gpt-4.1-mini-2025-04-14",
        "is_reasoning_model": False,
        "price_in":           0.40,
        "price_out":          1.60,
        "score_col":          "DW_gpt41_mini_score",
        "max_tokens":         30,
    },
    "gpt41_nano": {
        "enabled":            True,
        "model":              "gpt-4.1-nano-2025-04-14",
        "is_reasoning_model": False,
        "price_in":           0.10,
        "price_out":          0.40,
        "score_col":          "DW_gpt41_nano_score",
        "max_tokens":         30,
    },
}

# =========================
# JSON SCHEMA
# =========================
DW_JSON_SCHEMA = {
    "format": {
        "type":   "json_schema",
        "name":   "dw_score",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "score": {"type": "integer"},
            },
            "required":             ["score"],
            "additionalProperties": False,
        },
    }
}

# =========================
# PROMPTS
# =========================
SYSTEM_PROMPT = """
You will analyze a passage from a political speech to determine whether it suggests collusion by powerful actors against the American people.

Collusion refers to secretive coordination to manipulate events or undermine public interests of United States.
Only consider collusion relevant if it concerns the American public, U.S. institutions, or U.S. political, economic, or social interests.

Evaluate the passage based on the following scale:
1. The passage does not mention or imply any collusion.
2. The passage mentions powerful actors but does not suggest collusion, or it explicitly rejects the idea of collusion.
3. The passage refers to powerful actors and could potentially suggest collusion, but the implication is weak or ambiguous.
4. The passage strongly implies that powerful actors are engaged in collusion to manipulate events or undermine public interests.
5. The passage explicitly asserts that powerful actors are engaged in collusion to secretly influence events or act against the public interest.

### Additional Considerations:
- Consider that references to powerful actors or coded language may subtly imply collusion, even if it is not explicitly stated.
- If the paragraph uses phrases or rhetorical cues often associated with conspiracy or secrecy, treat these as indicators of collusion.
- If the speaker alludes to hidden motives or behind-the-scenes coordination, assume this suggests collusion.
- If the passage contains a direct quote or secondary source, only assign it to the speaker if they appear to endorse or align with that material. Analyze tone, intention, and how statements relate to each other to answer whether the speaker believes in collusion or criticizes someone else claiming it.
- Consider collusion in a strict sense, meaning that there are multiple powerful actors for whom coordination is costly.
- Consider that if a speaker quotes someone from a different political party, they may be disagreeing with or criticizing the quoted content. Be attentive to party dynamics in such cases.

Return VALID JSON ONLY in EXACTLY this format (no markdown, no extra text):
{"score": <integer between 1 and 5 inclusive>}
""".strip()

USER_TEMPLATE = """
The speaker is affiliated with the {party} party.
If the paragraph quotes someone from a different political party, the speaker might disagree with the quoted content.

Passage:
"{passage}"
""".strip()

# =========================
# RUN STATS
# =========================
RUN_STATS = {
    name: Counter({"ok": 0, "repaired": 0, "failed": 0,
                   "api_error": 0, "input_tokens": 0, "output_tokens": 0})
    for name in MODEL_CFG.keys()
}

# =========================
# HELPERS
# =========================
def safe_test_output_path(path: str) -> str:
    root, ext = os.path.splitext(path)
    return f"{root}{TEST_OUTPUT_SUFFIX}{ext}"

def _read_tabular(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm", ".xlsb"):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def save_full(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_excel(path, index=False)

def build_user_message(row: pd.Series) -> str:
    party   = (str(row.get("party", "")) or "UNKNOWN").strip()
    passage = str(row.get("paragraph_clean", "")).strip()
    return USER_TEMPLATE.format(party=party, passage=passage)

def enabled_models() -> Dict[str, Dict]:
    return {k: v for k, v in MODEL_CFG.items() if v.get("enabled", True)}

def all_output_cols() -> List[str]:
    return [cfg["score_col"] for cfg in MODEL_CFG.values()]

def ensure_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in all_output_cols():
        if c not in df.columns:
            df[c] = ""
    return df

def row_done(df: pd.DataFrame, i: int, model_name: str) -> bool:
    return str(df.at[i, MODEL_CFG[model_name]["score_col"]]).strip() != ""

def resume_merge(df: pd.DataFrame, out_path: str) -> pd.DataFrame:
    if not os.path.exists(out_path):
        return ensure_cols(df)
    old = _read_tabular(out_path)
    old = ensure_cols(old)
    df  = ensure_cols(df)
    key = None
    for k in ["q_id", "ParagraphID"]:
        if k in df.columns and k in old.columns:
            key = k
            break
    if key is None:
        return df
    df[key]  = df[key].astype(str).str.strip()
    old[key] = old[key].astype(str).str.strip()
    old = old.sort_values(key).drop_duplicates(key, keep="last")
    keep_cols = [key] + all_output_cols()
    old = old[[c for c in keep_cols if c in old.columns]]
    df  = df.merge(old, on=key, how="left", suffixes=("", "_old"))
    for c in all_output_cols():
        oldc = f"{c}_old"
        if oldc in df.columns:
            df[c] = df[c].where(df[c].astype(str).str.strip() != "", df[oldc].fillna(""))
            df.drop(columns=[oldc], inplace=True)
    return df

# =========================
# LOGGING
# =========================
def save_inference_log(
    model_name:    str,
    row_idx:       int,
    user_msg:      str,
    raw_output:    str,
    status:        str,
    broken_output: Optional[str] = None,
    note:          Optional[str] = None,
) -> None:
    ts  = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    obj = {
        "model":         model_name,
        "row_index":     row_idx,
        "timestamp_utc": ts,
        "status":        status,
        "user_prompt":   user_msg,
        "raw_output":    raw_output,
    }
    if broken_output is not None:
        obj["broken_output"] = broken_output
    if note is not None:
        obj["note"] = note

    fname = f"{model_name}_row{row_idx:05d}_{status}_{ts}.json"
    with open(os.path.join(RUN_LOG_DIR, fname), "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

# =========================
# JSON PARSING & VALIDATION
# =========================
def extract_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = str(text).strip()
    if t.startswith("```"):
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
        if t.startswith("json"):
            t = t[4:]
        t = t.strip()
    try:
        obj = json.loads(t)
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        if isinstance(obj, str) and obj.strip().startswith("{"):
            obj = json.loads(obj)
        return obj
    except Exception:
        return None

def code_repair_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = re.sub(r"```json|```", "", str(text)).strip()
    m = re.search(r"\{.*", t, flags=re.DOTALL)
    if not m:
        return None
    t = m.group(0)
    t = re.sub(r",\s*([}\]])", r"\1", t)
    if t.count('"') % 2 != 0:
        t += '"'
    t += "}" * max(0, t.count("{") - t.count("}"))
    t += "]" * max(0, t.count("[") - t.count("]"))
    try:
        return json.loads(t)
    except Exception:
        pass
    score_m = re.search(r'"score"\s*:\s*([1-5])', t)
    if score_m:
        return {"score": int(score_m.group(1))}
    return None

def validate_dw(obj: Any) -> Tuple[bool, str, Optional[int]]:
    if not isinstance(obj, dict):
        return False, "Not a dict", None
    s = obj.get("score")
    if isinstance(s, str) and s.strip().isdigit():
        s = int(s.strip())
    if not isinstance(s, int) or not (1 <= s <= 5):
        return False, f"Bad score: {s!r}", None
    return True, "OK", s

# =========================
# RETRY LOGIC
# =========================
async def with_retries(call_fn, meta: str, max_retries: int = 10,
                       base_wait: float = 2, max_wait: float = 120):
    last_err = None
    for i in range(max_retries):
        try:
            return await call_fn()
        except Exception as e:
            last_err = e
            msg = str(e)
            low = msg.lower()
            is_429       = "429" in low or "rate limit" in low
            is_transient = (
                is_429 or
                "503" in low or "overload" in low or "unavailable" in low or
                "timeout" in low or "timed out" in low or "deadline" in low or
                "temporarily" in low or "try again" in low
            )
            if not is_transient:
                return f"Error:{msg[:300]}"
            base = 15 if is_429 else base_wait
            wait = min(max_wait, base * (2 ** i)) + random.uniform(0, 1.5)
            print(f"\n⚠️  Transient error ({meta}): {msg[:120]} | retry in {wait:.1f}s")
            await asyncio.sleep(wait)
    return f"Error:MaxRetries:{meta}:{str(last_err)[:200]}"

# =========================
# API CALLS
# =========================
async def call_model(
    model_id:            str,
    user_msg:            str,
    is_reasoning_model:  bool,
    max_tokens:          int,
) -> Tuple[str, int, int]:
    kwargs: Dict[str, Any] = dict(
        model=model_id,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        text=DW_JSON_SCHEMA,
        max_output_tokens=max_tokens,
    )
    if is_reasoning_model:
        kwargs["reasoning"] = {"effort": "low"}
    else:
        kwargs["temperature"] = TEMPERATURE

    resp       = await client.responses.create(**kwargs)
    text       = (resp.output_text or "").strip()
    in_tokens  = resp.usage.input_tokens  if resp.usage else 0
    out_tokens = resp.usage.output_tokens if resp.usage else 0
    return text, in_tokens, out_tokens


async def call_model_repair(
    model_id:           str,
    repair_system:      str,
    repair_user:        str,
    is_reasoning_model: bool,
    max_tokens:         int,
) -> Tuple[str, int, int]:
    kwargs: Dict[str, Any] = dict(
        model=model_id,
        input=[
            {"role": "system", "content": repair_system},
            {"role": "user",   "content": repair_user},
        ],
        max_output_tokens=max_tokens,
    )
    if is_reasoning_model:
        kwargs["reasoning"] = {"effort": "none"}
    else:
        kwargs["temperature"] = 0

    resp       = await client.responses.create(**kwargs)
    text       = (resp.output_text or "").strip()
    in_tokens  = resp.usage.input_tokens  if resp.usage else 0
    out_tokens = resp.usage.output_tokens if resp.usage else 0
    return text, in_tokens, out_tokens

# =========================
# INFERENCE
# =========================
async def score_dw(
    model_name: str,
    row:        pd.Series,
    sema:       asyncio.Semaphore,
    meta:       str,
    row_idx:    int,
) -> Optional[int]:
    async with sema:
        user_msg           = build_user_message(row)
        cfg                = MODEL_CFG[model_name]
        model_id           = cfg["model"]
        is_reasoning_model = cfg["is_reasoning_model"]
        max_tokens         = cfg["max_tokens"]

        # ── Primary call ──────────────────────────────────────────────
        async def _primary():
            return await call_model(model_id, user_msg, is_reasoning_model, max_tokens)

        res_tuple = await with_retries(_primary, meta=meta)
        if isinstance(res_tuple, str) and res_tuple.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res_tuple, "api_error")
            return None

        res, in_tok, out_tok = res_tuple
        RUN_STATS[model_name]["input_tokens"]  += in_tok
        RUN_STATS[model_name]["output_tokens"] += out_tok

        obj = extract_json_from_text(res) or code_repair_json(res)
        if obj is not None:
            ok, _, s = validate_dw(obj)
            if ok:
                RUN_STATS[model_name]["ok"] += 1
                return s

        # ── Repair call ───────────────────────────────────────────────
        if not ENABLE_REPAIR_CALL:
            RUN_STATS[model_name]["failed"] += 1
            save_inference_log(model_name, row_idx, user_msg, res, "failed",
                               note="Repair disabled.")
            return None

        repair_system   = (
            "Fix the following output into VALID JSON ONLY.\n"
            'Required format: {"score": 3}\n'
            "No markdown. No extra text. No extra keys."
        )
        repair_user_msg = f"Broken output:\n{res}"

        async def _repair():
            return await call_model_repair(
                model_id, repair_system, repair_user_msg, is_reasoning_model, max_tokens
            )

        res2_tuple = await with_retries(_repair, meta=meta + "|repair")
        if isinstance(res2_tuple, str) and res2_tuple.startswith("Error:"):
            RUN_STATS[model_name]["api_error"] += 1
            save_inference_log(model_name, row_idx, user_msg, res2_tuple, "api_error",
                               broken_output=res)
            return None

        res2, in_tok2, out_tok2 = res2_tuple
        RUN_STATS[model_name]["input_tokens"]  += in_tok2
        RUN_STATS[model_name]["output_tokens"] += out_tok2

        obj2 = extract_json_from_text(res2) or code_repair_json(res2)
        if obj2 is not None:
            ok2, _, s2 = validate_dw(obj2)
            if ok2:
                RUN_STATS[model_name]["repaired"] += 1
                save_inference_log(model_name, row_idx, user_msg, res2, "repaired",
                                   broken_output=res)
                return s2

        RUN_STATS[model_name]["failed"] += 1
        save_inference_log(model_name, row_idx, user_msg, res, "failed",
                           note="Failed after repair attempt.")
        return None

# =========================
# SEQUENTIAL RUNNER
# =========================
async def run_one_model(
    df:              pd.DataFrame,
    model_name:      str,
    sema:            asyncio.Semaphore,
    process_indices: List[int],
) -> None:
    cfg       = MODEL_CFG[model_name]
    score_col = cfg["score_col"]

    for k in range(0, len(process_indices), BATCH_SIZE):
        batch = process_indices[k:k + BATCH_SIZE]
        todo  = [i for i in batch if not row_done(df, i, model_name)]
        if not todo:
            continue

        async def run_row(ii: int):
            meta  = f"{model_name}|row={ii}"
            score = await score_dw(model_name, df.iloc[ii], sema, meta, ii)
            return ii, score

        tasks     = [asyncio.create_task(run_row(i)) for i in todo]
        batch_num = k // BATCH_SIZE + 1

        pbar = tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc=f"{model_name} batch {batch_num}",
            dynamic_ncols=True,
            leave=True,
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )

        for fut in pbar:
            i, score = await fut
            df.at[i, score_col] = "" if score is None else str(score)

        save_full(df, OUTPUT_PATH)
        tqdm.write(f"[Checkpoint] Saved → {OUTPUT_PATH}")

    in_cost, out_cost, total = compute_cost(model_name)
    stats   = RUN_STATS[model_name]
    in_tok  = stats["input_tokens"]
    out_tok = stats["output_tokens"]

    tqdm.write(f"\n{'='*55}")
    tqdm.write(f"  ✅ FINISHED: {model_name}  [{cfg['model']}]")
    tqdm.write(f"{'='*55}")
    tqdm.write(f"  Tokens   — input:  {in_tok:>12,}")
    tqdm.write(f"             output: {out_tok:>12,}")
    tqdm.write(f"  Prices   — input:  ${cfg['price_in']:.2f}/1M   output: ${cfg['price_out']:.2f}/1M")
    tqdm.write(f"  Cost     — input:  ${in_cost:>10.4f}")
    tqdm.write(f"             output: ${out_cost:>10.4f}")
    tqdm.write(f"             TOTAL:  ${total:>10.4f}")
    tqdm.write(f"{'='*55}\n")

# =========================
# COST & SUMMARY
# =========================
def compute_cost(model_name: str) -> Tuple[float, float, float]:
    cfg     = MODEL_CFG[model_name]
    stats   = RUN_STATS[model_name]
    in_tok  = stats["input_tokens"]
    out_tok = stats["output_tokens"]
    in_cost  = in_tok  / 1_000_000 * cfg["price_in"]
    out_cost = out_tok / 1_000_000 * cfg["price_out"]
    return in_cost, out_cost, in_cost + out_cost

def save_summary_log() -> None:
    ts    = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    fpath = os.path.join(RUN_LOG_DIR, f"summary_{ts}.txt")
    lines = ["=" * 65, "  FINAL SUMMARY — ALL MODELS", "=" * 65]

    grand_in_cost = grand_out_cost = grand_total = 0.0

    for model_name, cfg in enabled_models().items():
        stats      = RUN_STATS[model_name]
        ok         = stats["ok"]
        repaired   = stats["repaired"]
        failed     = stats["failed"]
        api_error  = stats["api_error"]
        in_tok     = stats["input_tokens"]
        out_tok    = stats["output_tokens"]
        total_rows = ok + repaired + failed + api_error
        in_cost, out_cost, total_cost = compute_cost(model_name)

        grand_in_cost  += in_cost
        grand_out_cost += out_cost
        grand_total    += total_cost

        lines.append(f"\n  {model_name}  [{cfg['model']}]")
        lines.append(f"    rows processed : {total_rows:,}  (ok={ok}, repaired={repaired}, failed={failed}, api_error={api_error})")
        lines.append(f"    input  tokens  : {in_tok:,}  → ${in_cost:.4f}")
        lines.append(f"    output tokens  : {out_tok:,}  → ${out_cost:.4f}")
        lines.append(f"    model cost     : ${total_cost:.4f}")

    lines += [
        f"\n{'─' * 65}",
        "  GRAND TOTAL",
        f"    input cost  : ${grand_in_cost:.4f}",
        f"    output cost : ${grand_out_cost:.4f}",
        f"    TOTAL COST  : ${grand_total:.4f}",
        f"\n  Run timestamp : {ts}",
        f"  Log folder    : {RUN_LOG_DIR}",
        f"  Output file   : {OUTPUT_PATH}",
        "=" * 65,
    ]

    with open(fpath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"  Summary saved → {fpath}")

def print_summary() -> None:
    print("\n" + "=" * 65)
    print("  FINAL SUMMARY — ALL MODELS")
    print("=" * 65)

    grand_in_cost = grand_out_cost = grand_total = 0.0

    for model_name, cfg in enabled_models().items():
        stats      = RUN_STATS[model_name]
        ok         = stats["ok"]
        repaired   = stats["repaired"]
        failed     = stats["failed"]
        api_error  = stats["api_error"]
        in_tok     = stats["input_tokens"]
        out_tok    = stats["output_tokens"]
        total_rows = ok + repaired + failed + api_error
        in_cost, out_cost, total_cost = compute_cost(model_name)

        grand_in_cost  += in_cost
        grand_out_cost += out_cost
        grand_total    += total_cost

        print(f"\n  {model_name}  [{cfg['model']}]")
        print(f"    rows processed : {total_rows:,}  (ok={ok}, repaired={repaired}, failed={failed}, api_error={api_error})")
        print(f"    input  tokens  : {in_tok:,}  → ${in_cost:.4f}")
        print(f"    output tokens  : {out_tok:,}  → ${out_cost:.4f}")
        print(f"    model cost     : ${total_cost:.4f}")

    print(f"\n{'─' * 65}")
    print("  GRAND TOTAL")
    print(f"    input cost  : ${grand_in_cost:.4f}")
    print(f"    output cost : ${grand_out_cost:.4f}")
    print(f"    TOTAL COST  : ${grand_total:.4f}")
    print(f"\n  Log folder: {RUN_LOG_DIR}")
    print("=" * 65 + "\n")
    save_summary_log()

# =========================
# MAIN
# =========================
async def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(INPUT_FILE)

    global OUTPUT_PATH, client
    OUTPUT_PATH = safe_test_output_path(OUTPUT_FILE) if DEBUG_TEST_ONLY else OUTPUT_FILE

    with open(OPENAI_KEY_PATH, "r", encoding="utf-8") as f:
        api_key = f.read().strip()
    if not api_key:
        raise RuntimeError("openai_api.txt is empty")
    client = AsyncOpenAI(api_key=api_key)

    df = _read_tabular(INPUT_FILE)
    df = ensure_cols(df)

    if RESUME_IF_OUTPUT_EXISTS and os.path.exists(OUTPUT_PATH):
        df = resume_merge(df, OUTPUT_PATH)

    for c in ["paragraph_clean", "party"]:
        if c not in df.columns:
            raise KeyError(f"Missing required column: {c}")

    active_models = enabled_models()
    if not active_models:
        raise ValueError("No models enabled in MODEL_CFG.")

    n               = len(df)
    process_indices = list(range(min(DEBUG_N_TEST, n))) if DEBUG_TEST_ONLY else list(range(n))
    sema            = asyncio.Semaphore(CONCURRENCY)

    print(f"Rows: {n:,} | Processing: {len(process_indices):,}")
    print(f"Output:     {OUTPUT_PATH}")
    print(f"Run logs:   {RUN_LOG_DIR}")
    print(f"Test mode:  {DEBUG_TEST_ONLY}")
    print(f"Enabled models: {list(active_models.keys())}")

    for model_name, cfg in active_models.items():
        print(f"\n=== Running {model_name}  [{cfg['model']}] ===")
        await run_one_model(df, model_name, sema, process_indices)

    save_full(df, OUTPUT_PATH)
    print_summary()
    print("✅ DONE")

# RUN
await main()

Rows: 511 | Processing: 511
Output:     /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/output/speech_classification_results_base_multi_models_2.xlsx
Run logs:   /Users/ghost/Library/CloudStorage/Dropbox/Conspiratorial_speech/Ali/fine_tuning_models/temp/_openai_inference_logs/run_20260406T151927
Test mode:  False
Enabled models: ['gpt54', 'gpt54_mini', 'gpt54_nano', 'gpt41', 'gpt41_mini', 'gpt41_nano']

=== Running gpt54  [gpt-5.4-2026-03-05] ===

  ✅ FINISHED: gpt54  [gpt-5.4-2026-03-05]
  Tokens   — input:             0
             output:            0
  Prices   — input:  $2.50/1M   output: $15.00/1M
  Cost     — input:  $    0.0000
             output: $    0.0000
             TOTAL:  $    0.0000


=== Running gpt54_mini  [gpt-5.4-mini-2026-03-17] ===

  ✅ FINISHED: gpt54_mini  [gpt-5.4-mini-2026-03-17]
  Tokens   — input:             0
             output:            0
  Prices   — input:  $0.75/1M   output: $4.50/1M
  Cost     — input:  $  